In [ ]:
!pip install -U huggingface_hub pyspellchecker

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")

In [ ]:
from huggingface_hub import login
login(hf_token)

In [ ]:
# ==========================================================
# CACHE CONFIG  -  reuse the fine-tuned BERT instead of retraining
# ==========================================================
# The ~15 min cost is the dataset-augmentation cell + the BERT training
# loop below. After training once, the model is pushed to the Hugging
# Face Hub; on every later run it is pulled back and both of those cells
# skip themselves.
#
#   FORCE_RETRAIN = False   -> ignore all caches, rebuild dataset + retrain
#   FORCE_RETRAIN = False  -> reuse the cached model whenever one exists
# ==========================================================
# RUN SELECTOR  -  change ONE line per run, then Save & Run All
#   "main"           -> soft targets ON,  contrastive ON,  upweight 6.0
#   "soft_off"       -> soft targets OFF, contrastive ON,  upweight 6.0
#   "no_contrastive" -> soft targets ON,  contrastive OFF, upweight 6.0
#   "soft_uw1"       -> soft targets ON,  contrastive ON,  upweight 1.0
#
# soft_uw1 exists to SEPARATE the two things the soft-label arm changes at
# once: the target representation (distribution vs majority vote) and the
# disagreement upweighting. main vs soft_uw1 isolates the weighting;
# soft_uw1 vs soft_off isolates the targets. Without it the soft-label
# ablation is a two-factor intervention reported as one.
# ==========================================================
RUN_LABEL  = "soft_off"      # <<<<<< CHANGE THIS PER RUN <<<<<<
TRAIN_SEED = 0           # <<<<<< AND THIS: 0, 1, 2 <<<<<<
#
# TRAIN_SEED controls ONLY training stochasticity (weight init, shuffling,
# dropout). The data split is governed by SPLIT_SEED below and is held FIXED at
# 42 for every arm and every seed, so all 12 runs are evaluated on byte-identical
# dev and test partitions. One seed per arm cannot show whether a
# difference survives training randomness; three can.

_RUN_CFG = {
    "main":           dict(soft=True,  contrastive=True,  upweight=6.0),
    "soft_off":       dict(soft=False, contrastive=True,  upweight=6.0),
    "no_contrastive": dict(soft=True,  contrastive=False, upweight=6.0),
    "soft_uw1":       dict(soft=True,  contrastive=True,  upweight=1.0),
}
if RUN_LABEL not in _RUN_CFG:
    raise ValueError(f"RUN_LABEL must be one of {list(_RUN_CFG)}, got {RUN_LABEL!r}")

SOFT_LABEL_TRAINING = _RUN_CFG[RUN_LABEL]["soft"]   # HateXplain per-annotator label distributions
                             # (class-weighted soft cross-entropy), TESTING whether the
                             # probability bars end up modelling annotator disagreement.
                             # So far they do not (entropy_corr ~ 0.06 - see SOFT-LABEL
                             # ANALYSIS cell): report this as a negative result, not a
                             # "spectrum" claim, unless a later run changes that.
                             # False -> original hard-label CE.
SOFT_LABEL_ALPHA    = 1.0    # 1.0 = pure soft CE; < 1 blends in hard CE on the majority label
SOFT_DISAGREEMENT_UPWEIGHT = _RUN_CFG[RUN_LABEL]["upweight"]
                                   # weight on rows where annotators actually disagreed.
                                   # Driven by the RUN SELECTOR: 6.0 for main/soft_off/
                                   # no_contrastive, 1.0 for soft_uw1. Comparing main
                                   # against soft_uw1 isolates this factor from the
                                   # target representation.
#
# Point MODEL_REPO at a repo under YOUR Hugging Face account
# (the login cell above must have used a write token).
# ==========================================================
# ----------------------------------------------------------
# ENCODER INITIALISATION
# tum-nlp/bert-hateXplain was fine-tuned ON HateXplain, and 20,109 HateXplain
# rows are in our corpus - including part of our held-out test split. Starting
# from it therefore contaminates the in-distribution score that the whole
# model-selection comparison rests on. bert-base-uncased has no exposure to any
# text we evaluate on. The off-the-shelf hateXplain checkpoint is still
# evaluated, as an EXTERNAL baseline, in the BASELINES cell.
# ----------------------------------------------------------
BASE_MODEL = "bert-base-uncased"

FORCE_RETRAIN = True     # MUST stay True for every ablation run
USE_CONTRASTIVE = _RUN_CFG[RUN_LABEL]["contrastive"]   # driven by the RUN SELECTOR above
MODEL_REPO = "ARC-0xFF3/toxicity-bert-hatexplain"   # <-- change to your HF repo id
LOCAL_MODEL_DIR = "best_model"

import os
from pathlib import Path

MODEL_CACHED = False
if not FORCE_RETRAIN:
    if Path(LOCAL_MODEL_DIR, "config.json").exists():
        MODEL_CACHED = True
        print(f"Found local ./{LOCAL_MODEL_DIR} - reusing it, no training needed.")
    else:
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=MODEL_REPO, local_dir=LOCAL_MODEL_DIR)
            MODEL_CACHED = Path(LOCAL_MODEL_DIR, "config.json").exists()
            if MODEL_CACHED:
                print(f"Pulled fine-tuned model from HF Hub: {MODEL_REPO}")
        except Exception as e:
            print(f"No cached model on the Hub ({e!r}) - will train from scratch.")

print("MODEL_CACHED =", MODEL_CACHED, "| FORCE_RETRAIN =", FORCE_RETRAIN)

import glob as _glob

def _kaggle_input(filename, *fallbacks):
    """Locate a data file under /kaggle/input regardless of the mount path.
    Kaggle mounts a dataset at /kaggle/input/<slug>/, NOT /kaggle/input/datasets/<user>/<slug>/."""
    for cand in (_glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
                 + list(fallbacks) + [filename]):
        if os.path.exists(cand):
            return cand
    return filename


SPLIT_SEED = 42
SPLIT_HOLDOUT_FRAC = 0.20   # dev + test together; split 50/50 -> 80 / 10 / 10

def load_eval_split(which):
    """'dev' or 'test' from the 3-way stratified split. Prefers the CSVs the
    training cell writes; otherwise reconstructs them deterministically."""
    import pandas as _pd
    _p = f"/kaggle/working/{which}_split.csv"
    if os.path.exists(_p):
        return _pd.read_csv(_p)
    from sklearn.model_selection import train_test_split as _tts
    _d = _pd.read_csv("/kaggle/working/augmented_dataset.csv").dropna(subset=["text", "clean_label"])
    _d["clean_label"] = _d["clean_label"].astype(int)
    _tr, _h = _tts(_d, test_size=SPLIT_HOLDOUT_FRAC, random_state=SPLIT_SEED, stratify=_d["clean_label"])
    _dev, _te = _tts(_h, test_size=0.50, random_state=SPLIT_SEED, stratify=_h["clean_label"])
    return _dev if which == "dev" else _te


# RUN_LABEL is set in the RUN SELECTOR block at the top of this cell.
RUN_METRICS_PATH = "/kaggle/working/run_metrics.json"   # keyed by run_id = RUN_LABEL_sSEED
try:
    os.remove(RUN_METRICS_PATH)  # fresh row per run
except FileNotFoundError:
    pass

def record_metrics(**kw):
    """Merge results into run_metrics.json (the ablation ledger reads this)."""
    import json as _j
    m = {}
    if os.path.exists(RUN_METRICS_PATH):
        try: m = _j.load(open(RUN_METRICS_PATH))
        except Exception: m = {}
    for k, v in kw.items():
        try: m[k] = float(v)
        except (TypeError, ValueError): m[k] = v
    _j.dump(m, open(RUN_METRICS_PATH, "w"), indent=2)
    return m

record_metrics(run_label=RUN_LABEL,
               soft_label_training=int(SOFT_LABEL_TRAINING),
               soft_label_alpha=SOFT_LABEL_ALPHA,
               soft_disagreement_upweight=SOFT_DISAGREEMENT_UPWEIGHT,
               use_contrastive=int(USE_CONTRASTIVE),
               base_model=BASE_MODEL,
               train_seed=TRAIN_SEED,
               split_seed=SPLIT_SEED,
               run_id=f"{RUN_LABEL}_s{TRAIN_SEED}")

print(f"RUN_LABEL={RUN_LABEL} | TRAIN_SEED={TRAIN_SEED} | SOFT={SOFT_LABEL_TRAINING} | "
      f"CONTRASTIVE={USE_CONTRASTIVE} | UPWEIGHT={SOFT_DISAGREEMENT_UPWEIGHT} | "
      f"BASE={BASE_MODEL} | SPLIT_SEED={SPLIT_SEED} | FORCE_RETRAIN={FORCE_RETRAIN}")


In [ ]:
# --- cache guard: skip the dataset rebuild when a trained model is already available ---
if MODEL_CACHED:
    raise SystemExit("MODEL_CACHED -> skipping dataset augmentation (set FORCE_RETRAIN=True to rebuild).")
if not FORCE_RETRAIN and os.path.exists("/kaggle/working/augmented_dataset.csv"):
    raise SystemExit("augmented_dataset.csv already present -> skipping rebuild (FORCE_RETRAIN=True to force).")

# ==========================================================
# AUGMENT DATASET: hateXplain + ToxiGen + Latent Hatred + Multi-Sarcasm
# Run this in Colab/Kaggle (needs internet + `pip install datasets`)
# ==========================================================

# pip install datasets   # uncomment if not already installed

import pandas as pd
import numpy as np
from datasets import load_dataset

LABELS = {"hate speech": 0, "offensive": 1, "normal": 2}

# ------------------------------------------------------------------
# 0. Load your existing hateXplain CSV (the base dataset you already have)
# ------------------------------------------------------------------
# ---------------------------------------------------------------------------
# SAME-SOURCE ABLATION FIX
# Both the soft-label and hard-label arms are now built from the SAME HateXplain
# source (the annotator-level JSON). The hard arm simply collapses the 3-annotator
# distribution to its majority label. Previously the hard arm read a separate
# hateexplain.csv, so `soft_off` differed in BOTH loss and corpus (5,673 vs 6,306
# test rows) and the macro-F1 gap was not attributable to the loss alone.
# Set HX_SAME_SOURCE = False only to reproduce the old, confounded behaviour.
# ---------------------------------------------------------------------------
HX_SAME_SOURCE = True

if SOFT_LABEL_TRAINING or HX_SAME_SOURCE:
    # HateXplain WITH per-annotator labels: train on the empirical 3-annotator
    # label distribution instead of a single aggregated label -> the probability
    # bars then model disagreement, not just a point estimate.
    _HX_STR = {"hatespeech": 0, "normal": 1, "offensive": 2}   # HateXplain's own order
    HX_MAP  = {0: 0, 1: 2, 2: 1}                               # -> ours (0 hate, 1 offensive, 2 normal)

    def _hx_idx(lab):
        return int(lab) if isinstance(lab, (int, np.integer)) else _HX_STR[lab]

    def _load_hatexplain_annotated():
        # NOTE: both Hub attempts below are EXPECTED to fail on current `datasets`
        # versions - HF removed support for script-based datasets, and hatexplain
        # ships a hatexplain.py loader. The printed RuntimeError /
        # DatasetNotFoundError lines are informational, NOT a failure: the raw
        # GitHub JSON fallback immediately after is the working path and is what
        # produces the ~20k soft-labelled rows.
        for _kw in (dict(path="hatexplain", trust_remote_code=True),
                    dict(path="Paul/hatexplain", trust_remote_code=True)):
            try:
                _ds = load_dataset(**_kw)
                return [ex for sp in _ds for ex in _ds[sp]]
            except Exception as e:
                print("  ", _kw["path"], "unavailable:", repr(e)[:120])
        import requests
        _url = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json"
        _raw = requests.get(_url, timeout=60).json()
        return [{"post_tokens": r["post_tokens"],
                 "annotators": {"label": [a["label"] for a in r["annotators"]]}}
                for r in _raw.values()]

    _rows = []
    for _ex in _load_hatexplain_annotated():
        _v = np.zeros(3)
        for _lab in _ex["annotators"]["label"]:
            _v[HX_MAP[_hx_idx(_lab)]] += 1
        _v = _v / _v.sum()
        _rows.append((" ".join(_ex["post_tokens"]), int(_v.argmax()), _v[0], _v[1], _v[2]))
    base_df = (pd.DataFrame(_rows, columns=["text", "clean_label",
                                            "soft_hate", "soft_offensive", "soft_normal"])
                 .drop_duplicates(subset=["text"]).reset_index(drop=True))
    base_df["source"] = "hateXplain"
    _dis = int((base_df[["soft_hate", "soft_offensive", "soft_normal"]].max(axis=1) < 0.999).sum())
    print(f"Base HateXplain rows (with soft labels): {len(base_df)}  |  annotator-disagreement rows: {_dis}")

    if not SOFT_LABEL_TRAINING:
        # HARD-LABEL ARM, SAME SOURCE: keep clean_label (already the annotator
        # majority via argmax) and drop the soft columns so the training cell
        # falls back to plain cross-entropy. Identical rows, identical split,
        # only the loss differs.
        base_df = base_df.drop(columns=["soft_hate", "soft_offensive", "soft_normal"])
        print(f"  HARD-LABEL arm: soft columns dropped, {len(base_df)} rows kept "
              f"(same source as the soft arm - loss is the only difference)")
elif False:
    base_df = pd.read_csv(_kaggle_input("hateexplain.csv",
        "/kaggle/input/datasets/alinashifa/hateexplain-csv/hateexplain.csv"))
    base_df["clean_label"] = base_df["label"].str.lower().str.strip().map(LABELS)
    base_df = base_df.dropna(subset=["clean_label"]).reset_index(drop=True)
    base_df = base_df[["comment", "clean_label"]].rename(columns={"comment": "text"})
    base_df["source"] = "hateXplain"
    print("Base hateXplain rows:", len(base_df))

# ------------------------------------------------------------------
# 1. ToxiGen — implicit toxicity, no slurs, mentions minority groups
#    Fields: 'generation' (text), 'prompt_label' (1=toxic, 0=benign)
# ------------------------------------------------------------------
toxigen = load_dataset("toxigen/toxigen-data", "train")["train"]
print("ToxiGen columns:", toxigen.column_names)  # sanity check — adjust below if names differ

toxigen_df = toxigen.to_pandas()
text_col = "generation" if "generation" in toxigen_df.columns else "text"
label_col = "prompt_label" if "prompt_label" in toxigen_df.columns else "label"

toxigen_df = toxigen_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "tox_label"})
# prompt_label labels the PROMPT, not the generation: it agrees with human judgment for
# only ~68% of top-k toxic examples (~40% ALICE-decoded) but ~92-95% for benign. So the
# machine set contributes ONLY prompt_label == 0 -> normal (identity-mention FP traps);
# the toxic side comes entirely from the human-annotated ToxiGen_human subset below.
toxigen_df = toxigen_df[toxigen_df["tox_label"] == 0].reset_index(drop=True)
toxigen_df["clean_label"] = LABELS["normal"]
toxigen_df["source"] = "ToxiGen_machine_benign"
toxigen_df = toxigen_df[["text", "clean_label", "source"]]
# These are identity-mention false-positive traps, not the bulk of training data. Keeping
# all ~39k made the corpus 40% machine-benign and turned the model into a normal-detector
# (0 benign FPs, ~27% bullying recall). Subsample to ~human-subset scale.
TOXIGEN_MACHINE_BENIGN_N = 6000
toxigen_df = toxigen_df.sample(n=min(TOXIGEN_MACHINE_BENIGN_N, len(toxigen_df)),
                               random_state=42).reset_index(drop=True)
print(f"ToxiGen benign (prompt_label==0) rows kept: {len(toxigen_df)} "
      f"(capped at TOXIGEN_MACHINE_BENIGN_N={TOXIGEN_MACHINE_BENIGN_N})")

# 1b. ToxiGen HUMAN-annotated subset. prompt_label labels the *prompt*, not the
#     generation; the "annotated" config carries per-text human toxicity scores.
# 1-5 Likert mean across annotators. 3.0 is precisely where annotators split, so a
# single midpoint threshold forces the ambiguous middle into "toxic" and dumps mildly
# toxic 2.x rows into "normal". Two thresholds, middle band dropped, instead.
TOXIGEN_TOXIC_THRESH  = 4.0
TOXIGEN_BENIGN_THRESH = 2.0
try:
    _tga = load_dataset("toxigen/toxigen-data", "annotated")
    _tga = pd.concat([_tga[s].to_pandas() for s in _tga], ignore_index=True)
    if "toxicity_human" not in _tga.columns:
        raise KeyError(f"toxicity_human missing; got {list(_tga.columns)} - refusing to fall "
                       f"back to toxicity_ai (that IS the machine label this fix removes)")
    _tcol = "text" if "text" in _tga.columns else "generation"
    toxigen_human_df = _tga[[_tcol, "toxicity_human"]].rename(
        columns={_tcol: "text", "toxicity_human": "_th"})
    toxigen_human_df["_th"] = pd.to_numeric(toxigen_human_df["_th"], errors="coerce")
    toxigen_human_df = toxigen_human_df.dropna(subset=["_th"]).reset_index(drop=True)
    _pre = len(toxigen_human_df)
    _keep = ((toxigen_human_df["_th"] >= TOXIGEN_TOXIC_THRESH)
             | (toxigen_human_df["_th"] <= TOXIGEN_BENIGN_THRESH))
    toxigen_human_df = toxigen_human_df[_keep].reset_index(drop=True)
    toxigen_human_df["clean_label"] = np.where(
        toxigen_human_df["_th"] >= TOXIGEN_TOXIC_THRESH, LABELS["hate speech"], LABELS["normal"])
    toxigen_human_df["source"] = "ToxiGen_human"
    toxigen_human_df = toxigen_human_df[["text", "clean_label", "source"]].drop_duplicates("text")
    toxigen_df = toxigen_df[~toxigen_df["text"].isin(set(toxigen_human_df["text"]))].reset_index(drop=True)
    print(f"ToxiGen human rows: {len(toxigen_human_df)} kept (>= {TOXIGEN_TOXIC_THRESH} toxic, "
          f"<= {TOXIGEN_BENIGN_THRESH} benign) | {_pre - len(toxigen_human_df)} middle-band rows "
          f"dropped | machine-benign rows after dedup: {len(toxigen_df)}")
except Exception as _e:
    print("ToxiGen annotated subset unavailable:", repr(_e)[:150])
    toxigen_human_df = pd.DataFrame(columns=["text", "clean_label", "source"])

# ------------------------------------------------------------------
# 2. Latent Hatred — real tweets with coded/implicit hate language
#    Fields: 'post' (text), 'class' (implicit_hate / explicit_hate / not_hate)
# ------------------------------------------------------------------
latent = load_dataset("tasksource/implicit-hate-stg1")["train"]
print("Latent Hatred columns:", latent.column_names)  # sanity check

latent_df = latent.to_pandas()
text_col = "post" if "post" in latent_df.columns else "text"
label_col = "class" if "class" in latent_df.columns else "label"

latent_df = latent_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "lh_label"})
latent_label_map = {
    "implicit_hate": LABELS["hate speech"],
    "explicit_hate": LABELS["hate speech"],
    "not_hate": LABELS["normal"],
}
latent_df["clean_label"] = latent_df["lh_label"].map(latent_label_map)
latent_df = latent_df.dropna(subset=["clean_label"]).reset_index(drop=True)
latent_df["source"] = "LatentHatred"
latent_df = latent_df[["text", "clean_label", "source"]]
print("Latent Hatred rows:", len(latent_df))

# ------------------------------------------------------------------
# 3. Multi-Sarcasm -- DROPPED from training. The sarcasm_level -> toxicity tertile
#    mapping is an unvalidated labelling assumption; see the sarcasm-audit cell,
#    which keeps the 200-row audit sample as a released artifact.

# ------------------------------------------------------------------
# 3.5. GoEmotions — genuine positive/warm sentiment, to teach the model
#      what real appreciation/kindness looks like, not just "not flagged"
#      Fields: 'text', 'labels' (list of emotion label IDs)
# ------------------------------------------------------------------
goemotions = load_dataset("google-research-datasets/go_emotions", "simplified")["train"]
print("GoEmotions columns:", goemotions.column_names)

ge_label_names = goemotions.features["labels"].feature.names
print("GoEmotions label names sample:", ge_label_names[:10])

# Emotions that represent genuine warmth/appreciation -> reinforce as "normal"
POSITIVE_EMOTIONS = {"admiration", "gratitude", "joy", "love", "caring", "optimism", "approval", "pride"}
positive_label_ids = {i for i, name in enumerate(ge_label_names) if name in POSITIVE_EMOTIONS}

ge_df = goemotions.to_pandas()
ge_df["has_positive_emotion"] = ge_df["labels"].apply(
    lambda lbls: any(l in positive_label_ids for l in lbls)
)
ge_positive_df = ge_df[ge_df["has_positive_emotion"]][["text"]].copy()
ge_positive_df["clean_label"] = LABELS["normal"]
ge_positive_df["source"] = "GoEmotions_positive"

# Subsample to a reasonable size — this is reinforcement, not the bulk of training data
ge_positive_df = ge_positive_df.sample(
    n=min(8000, len(ge_positive_df)), random_state=42
).reset_index(drop=True)
print("GoEmotions positive rows:", len(ge_positive_df))
print("ASSUMPTION (unaudited): GoEmotions positive-emotion labels (admiration/gratitude/"
      "joy/love/caring/optimism/approval/pride) -> normal. Lower risk than the sarcasm "
      "mapping, but approval/optimism can co-occur with sarcasm; not spot-checked.")

# ------------------------------------------------------------------
# 3.6. Hand-written contrastive examples (meta-commentary vs. attack,
#      backhanded vs. genuine compliment, exclusion/concern/gaslighting
#      vs. their benign counterparts). Small, but targets specific
#      decision boundaries the bulk data doesn't cover.
#      Files expected to have columns: text, clean_label, category, rationale
# ------------------------------------------------------------------
contrastive_paths = [
    _kaggle_input("contrastive_examples_final.csv",
                  "/kaggle/input/datasets/alinashifa/contrastive-final/contrastive_examples_final.csv"),
]

PAIRED_CATEGORIES = [
    ("meta_commentary",            "direct_attack"),
    ("meta_commentary",            "direct_attack_same_topic"),
    ("untargeted_absurdist_humor", "targeted_attack"),
    ("genuine_warmth",             "sarcastic_insult"),
    ("genuine_compliment",         "backhanded_compliment"),
    ("genuine_compliment",         "sarcastic_insult"),
    ("neutral_statement",          "dogwhistle"),
    ("dogwhistle",                 "coded_offensive"),
    ("dark_humor_benign",          "dark_humor_targeted"),
    ("genuine_criticism",          "dismissive_insult"),
]


def _add_pair_ids(dfc):
    """pair_id for genuinely paired contrastive categories. Uses a (category, key)
    column pair if present, else falls back to ADJACENCY: consecutive rows whose
    categories are a known contrastive sibling pair get the same id. Unpaired rows
    stay blank."""
    if "pair_id" in dfc.columns:
        return dfc
    dfc = dfc.reset_index(drop=True).copy()
    dfc["pair_id"] = ""
    _cat = next((c for c in ("category", "cat", "type") if c in dfc.columns), None)
    _key = next((c for c in ("topic", "anchor", "pair", "ref", "ref_id", "pair_key") if c in dfc.columns), None)
    # a category can have more than one sibling (meta_commentary pairs with two attack
    # categories, genuine_compliment with two insult categories) -> frozensets, not a dict.
    _sib = {frozenset(p) for p in PAIRED_CATEGORIES}
    _flat = {c for pr in _sib for c in pr}
    if _cat and _key:
        _n = 0
        for _k, _g in dfc[dfc[_cat].astype(str).isin(_flat)].groupby(_key):
            if _g[_cat].nunique() >= 2:
                _n += 1
                dfc.loc[_g.index, "pair_id"] = f"P{_n:03d}"
        print(f"  pair_id: {_n} pairs from ({_cat}, {_key})")
    elif _cat:
        _cats = dfc[_cat].astype(str).tolist()
        _n, i = 0, 0
        while i < len(_cats) - 1:
            if frozenset((_cats[i], _cats[i + 1])) in _sib and not dfc.at[i, "pair_id"]:
                _n += 1
                dfc.at[i, "pair_id"] = dfc.at[i + 1, "pair_id"] = f"P{_n:03d}"
                i += 2
            else:
                i += 1
        _blank = int((dfc["pair_id"] == "").sum())
        print(f"  pair_id: {_n} pairs by adjacency on ({_cat}); {_blank} rows unpaired "
              f"(add a `topic` column by hand if this is 0)")
        if _n < 20:
            print(f"  !!! WARNING: only {_n} contrastive pairs found - check PAIRED_CATEGORIES "
                  f"against the actual category values: {sorted(set(_cats))}")
    else:
        print("  pair_id: left blank - no category column")
    return dfc


contrastive_dfs = []
if True:   # always load: USE_CONTRASTIVE is applied at TRAIN time, not corpus time
    for p in contrastive_paths:
        if not os.path.exists(p):
            print(f"Skipping {p} - not found")
            continue
        cdf_full = _add_pair_ids(pd.read_csv(p))
        cdf_full.to_csv("/kaggle/working/contrastive_released.csv", index=False)
        try:
            from huggingface_hub import upload_file
            upload_file(path_or_fileobj="/kaggle/working/contrastive_released.csv",
                        path_in_repo="contrastive_examples_final.csv", repo_id=MODEL_REPO)
        except Exception:
            pass
        cdf = cdf_full[["text", "clean_label"]].copy()
        cdf["clean_label"] = cdf["clean_label"].astype(int)
        cdf["source"] = "Contrastive_handwritten"
        contrastive_dfs.append(cdf)
        print(f"Loaded {p}: {len(cdf)} rows")
        break
    if not contrastive_dfs:
        print("!!! WARNING: USE_CONTRASTIVE=True but NO contrastive CSV was found - "
              "`main` and `no_contrastive` will be identical. Check the dataset slug / "
              "glob /kaggle/input/**/contrastive_examples_final.csv")
else:
    print("contrastive loader skipped")

if contrastive_dfs:
    contrastive_df = pd.concat(contrastive_dfs, ignore_index=True)
else:
    contrastive_df = pd.DataFrame(columns=["text", "clean_label", "source"])

print("Total contrastive rows:", len(contrastive_df))
if "record_metrics" in dir():
    record_metrics(contrastive_rows=len(contrastive_df),
                   contrastive_found=int(len(contrastive_dfs) > 0))

# ------------------------------------------------------------------
# 4. Merge everything into one augmented dataset
# ------------------------------------------------------------------
base_df["clean_label"] = base_df["clean_label"].astype(int)
toxigen_df["clean_label"] = toxigen_df["clean_label"].astype(int)
latent_df["clean_label"] = latent_df["clean_label"].astype(int)
ge_positive_df["clean_label"] = ge_positive_df["clean_label"].astype(int)

# label provenance: human annotation vs model-generated vs heuristic mapping
_PROV = {"hateXplain": "human", "ToxiGen": "machine", "ToxiGen_machine": "machine",
         "ToxiGen_machine_benign": "machine",
         "ToxiGen_human": "human", "LatentHatred": "human",
         "GoEmotions_positive": "heuristic"}
for _d in (base_df, toxigen_df, toxigen_human_df, latent_df, ge_positive_df, contrastive_df):
    if len(_d):
        _d["label_source"] = _d["source"].map(
            lambda s: _PROV.get(s, "human" if str(s).startswith("Contrastive") else "heuristic"))

# MASTER-SPLIT FIX : the contrastive rows are ALWAYS loaded and
# ALWAYS merged, so the corpus - and therefore the stratified split - is byte
# identical for every arm. USE_CONTRASTIVE no longer changes the corpus; it is
# applied later, in the training cell, by dropping the tagged rows from the TRAIN
# partition only. Dev and test are then identical across all arms and all seeds.
augmented_df = pd.concat(
    [base_df, toxigen_df, toxigen_human_df, latent_df, ge_positive_df, contrastive_df],
    ignore_index=True,
)
augmented_df["label_source"] = augmented_df["label_source"].fillna("heuristic")
augmented_df = augmented_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# soft targets: real annotator distributions for HateXplain, one-hot for every other source
if SOFT_LABEL_TRAINING:
    _sc = ["soft_hate", "soft_offensive", "soft_normal"]
    for _c in _sc:
        if _c not in augmented_df.columns:
            augmented_df[_c] = np.nan
    _need = augmented_df[_sc].isna().any(axis=1)
    augmented_df.loc[_need, _sc] = np.eye(3)[augmented_df.loc[_need, "clean_label"].astype(int).to_numpy()]
    print(f"Soft targets: {(~_need).sum()} annotator-derived, {int(_need.sum())} one-hot")


print("\n=== Final dataset summary ===")
print("Total rows:", len(augmented_df))
print(augmented_df["source"].value_counts())
print(augmented_df["label_source"].value_counts())
print(augmented_df["clean_label"].value_counts().rename(index={v: k for k, v in LABELS.items()}))

print("\nsource x label crosstab (class 'offensive' provenance concentration):")
print(pd.crosstab(augmented_df["source"],
                  augmented_df["clean_label"].map({0: "hate", 1: "offensive", 2: "normal"})))
print("LIMITATION: with MultiSarcasm dropped, ToxiGen/LatentHatred/GoEmotions only ever "
      "emit hate or normal - class 'offensive' comes almost entirely from hateXplain and "
      "the small contrastive set. This concentration is a real constraint on the 3-class "
      "model and will show up as weaker offensive-class F1/recall.")
if "record_metrics" in dir():
    _off_src = augmented_df.loc[augmented_df["clean_label"] == 1, "source"].value_counts()
    record_metrics(
        offensive_class_top_source=(str(_off_src.index[0]) if len(_off_src) else "none"),
        offensive_class_top_source_frac=(float(_off_src.iloc[0] / max(1, _off_src.sum()))
                                        if len(_off_src) else 0.0),
        toxigen_policy="human_toxic_machine_benign",
        toxigen_toxic_thresh=TOXIGEN_TOXIC_THRESH,
        toxigen_benign_thresh=TOXIGEN_BENIGN_THRESH)

augmented_df.to_csv("augmented_dataset.csv", index=False)
print("\nSaved -> augmented_dataset.csv")


In [ ]:
# ============================================================
# SARCASM MAPPING AUDIT  -  RETIRED (kept for provenance only)
# ============================================================
# MultiSarcasm was dropped from the dataset entirely after the audit sample showed
# the sarcasm_level tertile mapping labelled ordinary comments ("Verified :)",
# "Yay supply drops for MW2 Remastered") as OFFENSIVE. Nothing downstream reads
# from it, so there is nothing left to audit and no CSV to fill in.
#
# The 200-row audit sample is kept as the record of the drop decision.
# Re-enable only if MultiSarcasm is ever
# reintroduced.
# ------------------------------------------------------------
print("Sarcasm audit: RETIRED - MultiSarcasm is not used for training. "
      "No labelling required, nothing to run here.")


In [ ]:
# --- cache guard: skip training when a fine-tuned model is already available ---
if MODEL_CACHED:
    raise SystemExit(f"MODEL_CACHED -> using ./{LOCAL_MODEL_DIR}, skipping training (FORCE_RETRAIN=True to retrain).")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# ---- reproducibility: SPLIT_SEED governs the partition (fixed for all arms),
# ---- TRAIN_SEED governs training stochasticity (varied 0/1/2).
import random as _rnd
SPLIT_SEED  = int(globals().get("SPLIT_SEED", 42))
TRAIN_SEED  = int(globals().get("TRAIN_SEED", 0))
_rnd.seed(TRAIN_SEED); np.random.seed(TRAIN_SEED); torch.manual_seed(TRAIN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAIN_SEED)
try:
    from transformers import set_seed as _hf_set_seed; _hf_set_seed(TRAIN_SEED)
except Exception:
    pass
print(f"seeds -> SPLIT_SEED={SPLIT_SEED} (fixed)  TRAIN_SEED={TRAIN_SEED} (varied)")

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
def _f1s_full(y, p):
    from sklearn.metrics import f1_score as _f
    return _f(y, p, average='macro', labels=[0, 1, 2], zero_division=0)
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from torch.optim import AdamW

# ------------------------------------------------------------------
# 1. LOAD THE AUGMENTED DATASET
# ------------------------------------------------------------------
csv_path = "/kaggle/working/augmented_dataset.csv"  # HateXplain(soft)+ToxiGen(human-toxic/machine-benign)+LatentHatred+GoEmotions(+contrastive if present)
df = pd.read_csv(csv_path)
print("Columns:", df.columns.tolist())
print(df["clean_label"].value_counts())

LABELS = {"hate speech": 0, "offensive": 1, "normal": 2}
LABELS_INV = {v: k for k, v in LABELS.items()}

df = df.dropna(subset=["text", "clean_label"]).reset_index(drop=True)
df["clean_label"] = df["clean_label"].astype(int)

# --- soft-label setup (from the CACHE CONFIG cell) --------------------------
SOFT_LABEL_TRAINING = bool(globals().get("SOFT_LABEL_TRAINING", False))
SOFT_LABEL_ALPHA    = float(globals().get("SOFT_LABEL_ALPHA", 1.0))
SOFT_DISAGREEMENT_UPWEIGHT = float(globals().get("SOFT_DISAGREEMENT_UPWEIGHT", 1.0))
SOFT_COLS = ["soft_hate", "soft_offensive", "soft_normal"]
USE_SOFT  = SOFT_LABEL_TRAINING and all(c in df.columns for c in SOFT_COLS)
if USE_SOFT:
    df[SOFT_COLS] = df[SOFT_COLS].fillna(0.0).astype(float)
    _rs = df[SOFT_COLS].sum(axis=1).replace(0, 1.0)
    df[SOFT_COLS] = df[SOFT_COLS].div(_rs, axis=0)          # renormalise defensively
    _ndis = int((df[SOFT_COLS].max(axis=1) < 0.999).sum())
    print(f"Soft-label training ON (alpha={SOFT_LABEL_ALPHA}) | {_ndis} rows carry annotator disagreement")
else:
    print("Soft-label training OFF -> hard-label class-weighted cross-entropy")

# ------------------------------------------------------------------
# 2. TRAIN / VALIDATION SPLIT (stratified so class balance is preserved)
# ------------------------------------------------------------------
train_df, _hold = train_test_split(
    df, test_size=0.20, random_state=SPLIT_SEED, stratify=df["clean_label"]
)
val_df, test_df = train_test_split(          # val_df is the DEV set: checkpoint / temperature / threshold
    _hold, test_size=0.50, random_state=SPLIT_SEED, stratify=_hold["clean_label"]
)
# ---- MASTER-SPLIT ABLATION: apply USE_CONTRASTIVE to TRAIN only ----
# The corpus, and therefore the split, is identical for every arm. The
# no_contrastive ablation removes the 155 hand-written rows from the TRAINING
# partition after the split, so dev and test are byte-identical across all arms
# and all seeds. Any contrastive rows that landed in dev/test are removed from
# BOTH evaluation partitions in every arm, so no author-written text is ever
# scored and the evaluation sets stay identical.
_is_contr = lambda d: d["source"].astype(str).str.startswith("Contrastive")
_n_dev_c, _n_test_c = int(_is_contr(val_df).sum()), int(_is_contr(test_df).sum())
val_df  = val_df[~_is_contr(val_df)].reset_index(drop=True)
test_df = test_df[~_is_contr(test_df)].reset_index(drop=True)
print(f"  removed {_n_dev_c} dev / {_n_test_c} test contrastive rows "
      f"(done in EVERY arm -> identical eval sets)")
if not USE_CONTRASTIVE:
    _n0 = len(train_df)
    train_df = train_df[~_is_contr(train_df)].reset_index(drop=True)
    print(f"  USE_CONTRASTIVE=False -> dropped {_n0-len(train_df)} contrastive rows "
          f"from TRAIN only ({_n0} -> {len(train_df)})")
else:
    print(f"  USE_CONTRASTIVE=True  -> {int(_is_contr(train_df).sum())} contrastive rows kept in TRAIN")

val_df.to_csv("/kaggle/working/dev_split.csv", index=False)
test_df.to_csv("/kaggle/working/test_split.csv", index=False)
print(f"Train {len(train_df)} | Dev {len(val_df)} | Test {len(test_df)}   (test is scored once, at the very end)")
if "record_metrics" in dir():
    record_metrics(n_train=len(train_df), n_dev=len(val_df), n_test=len(test_df))

# ------------------------------------------------------------------
# 3. MODEL + TOKENIZER SETUP
# ------------------------------------------------------------------
model_checkpoint = globals().get("BASE_MODEL", "bert-base-uncased")
# NOTE: previously tum-nlp/bert-hateXplain. Changed to a clean base because that
# checkpoint was fine-tuned on HateXplain, which supplies 20,109 rows of this
# corpus including part of the held-out test split. Expect lower absolute
# scores; what matters is whether the
# ordering between configurations survives.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config = AutoConfig.from_pretrained(model_checkpoint, num_labels=3, output_attentions=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, config=config, ignore_mismatched_sizes=True, torch_dtype=torch.float32
)
model = model.float()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ------------------------------------------------------------------
# 4. TOKENIZE (longer max_length to fit Reddit/ToxiGen sentences)
# ------------------------------------------------------------------
MAX_LEN = 128

print("Tokenizing train set...")
train_encodings = tokenizer(
    list(train_df["text"]), truncation=True, padding=True, max_length=MAX_LEN
)
print("Tokenizing val set...")
val_encodings = tokenizer(
    list(val_df["text"]), truncation=True, padding=True, max_length=MAX_LEN
)
test_encodings = tokenizer(
    list(test_df["text"]), truncation=True, padding=True, max_length=MAX_LEN
)


class ToxicityDataset(Dataset):
    def __init__(self, encodings, labels, soft=None):
        self.encodings = encodings
        self.labels = labels
        self.soft = soft

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        if self.soft is not None:
            item["soft_labels"] = torch.tensor(self.soft[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


_train_soft = train_df[SOFT_COLS].to_numpy() if USE_SOFT else None
_val_soft   = val_df[SOFT_COLS].to_numpy() if USE_SOFT else None
train_dataset = ToxicityDataset(train_encodings, list(train_df["clean_label"]), _train_soft)
val_dataset = ToxicityDataset(val_encodings, list(val_df["clean_label"]), _val_soft)

_g = torch.Generator(); _g.manual_seed(TRAIN_SEED)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, generator=_g)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataset = ToxicityDataset(test_encodings, list(test_df["clean_label"]),
                               test_df[SOFT_COLS].to_numpy() if USE_SOFT else None)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ------------------------------------------------------------------
# 5. CLASS WEIGHTS (so the majority class doesn't dominate the loss)
# ------------------------------------------------------------------
class_weights_raw = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["clean_label"].values,
)
# Milder dampening than sqrt (0.5) — full "balanced" weighting caused
# over-flagging of benign text; sqrt swung too far the other way and
# caused under-flagging of real toxic/coded content. This run uses 0.5 (sqrt
# dampening); 0.75 is the other value tried - keep whichever the ablation supports.
DAMPEN_EXPONENT = 0.5
class_weights = class_weights_raw ** DAMPEN_EXPONENT
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Raw class weights:", class_weights_raw)
print(f"Dampened class weights (^{DAMPEN_EXPONENT}):", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
_logsoftmax = nn.LogSoftmax(dim=-1)

def compute_loss(logits, labels, soft=None):
    """Class-weighted hard CE, or class-weighted SOFT cross-entropy when annotator
    distributions are available. Rows whose annotators disagreed are upweighted by
    SOFT_DISAGREEMENT_UPWEIGHT so they are not drowned out by the one-hot rows.
    A one-hot soft target reduces exactly to weighted CE."""
    hard = criterion(logits, labels)
    if soft is None:
        return hard
    logp = _logsoftmax(logits)
    per_ex = -(class_weights.unsqueeze(0) * soft * logp).sum(dim=1)            # (B,)
    w = torch.where(soft.max(dim=1).values < 0.999,
                    per_ex.new_tensor(SOFT_DISAGREEMENT_UPWEIGHT),
                    per_ex.new_tensor(1.0))
    soft_ce = (per_ex * w).sum() / w.sum()
    return SOFT_LABEL_ALPHA * soft_ce + (1.0 - SOFT_LABEL_ALPHA) * hard
optim = AdamW(model.parameters(), lr=2e-5)

# ------------------------------------------------------------------
# 6. EVALUATION FUNCTION
# ------------------------------------------------------------------
def evaluate_disagreement(model, loader):
    """On val rows where annotators actually disagreed: how close is the model's
    predictive distribution to theirs, and does model entropy track human entropy?
    Diagnostic only - checkpoint selection stays on macro-F1."""
    model.eval()
    kl, h_human, h_model = [], [], []
    with torch.no_grad():
        for batch in loader:
            if "soft_labels" not in batch:
                return None
            q = batch["soft_labels"]
            nd = q.max(dim=1).values < 0.999
            if nd.sum() == 0:
                continue
            logits = model(input_ids=batch["input_ids"].to(device),
                           attention_mask=batch["attention_mask"].to(device)).logits.cpu()
            p = logits.softmax(dim=1)
            qn, pn = q[nd].clamp_min(1e-9), p[nd].clamp_min(1e-9)
            kl.extend((qn * (qn.log() - pn.log())).sum(1).tolist())
            h_human.extend((-(qn * qn.log()).sum(1)).tolist())
            h_model.extend((-(pn * pn.log()).sum(1)).tolist())
    if not kl:
        return None
    r = float(np.corrcoef(h_human, h_model)[0, 1]) if len(h_human) > 2 else float("nan")
    return {"n": len(kl), "kl_q_p": float(np.mean(kl)), "entropy_corr": r}



def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    report = classification_report(
        all_labels, all_preds, target_names=["hate speech", "offensive", "normal"], digits=3
    )
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return report, macro_f1


# ------------------------------------------------------------------
# 7. TRAINING LOOP (with checkpointing on best val macro-F1)
# ------------------------------------------------------------------
EPOCHS = 8
PATIENCE = 3  # stop if val macro-F1 doesn't improve for this many consecutive epochs
best_f1 = 0.0
best_model_path = "best_model"
epochs_no_improve = 0
_spectrum_epochs = []   # (epoch, entropy_corr, KL) - reported together so it's not cherry-picked

BENIGN_CONTROL_SENTENCES = [
    "You are a good person.",
    "Hey! Take care of yourself.",
    "I really appreciate your help today.",
    "You did a great job on this project.",
    "Have a wonderful day!",
    "Thank you so much for being there for me.",
    "Congratulations on your achievement!",
]

def check_benign_false_positives(model, tokenizer):
    model.eval()
    flagged = 0
    for text in BENIGN_CONTROL_SENTENCES:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        if pred != 2:  # not "normal"
            flagged += 1
    return flagged, len(BENIGN_CONTROL_SENTENCES)

TOXIC_CONTROL_SENTENCES = [
    "Wow, you're SO smart, I'm amazed you found the door.",
    "Oh sure, because that worked out SO well last time.",
    "Nice job, genius.",
    "If you want your comment to get noticed, you post sexist, racist, homophobic etc content that other people won't be able to resist responding to.",
    "Some groups just aren't built for this kind of work.",
]

def check_toxic_recall(model, tokenizer):
    model.eval()
    caught = 0
    for text in TOXIC_CONTROL_SENTENCES:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=1).item()
        if pred != 2:  # correctly NOT "normal"
            caught += 1
    return caught, len(TOXIC_CONTROL_SENTENCES)

print("\n🚀 Training on augmented dataset...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        optim.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        soft = batch["soft_labels"].to(device) if "soft_labels" in batch else None
        loss = compute_loss(logits, labels, soft)
        loss.backward()
        optim.step()

        total_loss += loss.item()
        if batch_idx % 200 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} finished. Avg train loss: {avg_loss:.4f}")

    report, macro_f1 = evaluate(model, val_loader)
    print(f"Dev macro-F1: {macro_f1:.4f}")
    print(report)

    flagged, total_benign = check_benign_false_positives(model, tokenizer)
    print(f"⚠️  Benign control check: {flagged}/{total_benign} benign sentences misflagged as non-Normal")

    caught, total_toxic = check_toxic_recall(model, tokenizer)
    print(f"⚠️  Toxic control check: {caught}/{total_toxic} sarcastic/coded toxic sentences correctly caught")

    if USE_SOFT:
        _sp = evaluate_disagreement(model, val_loader)
        if _sp:
            _spectrum_epochs.append((epoch + 1, _sp["entropy_corr"], _sp["kl_q_p"]))
            print(f"🌈 Spectrum check (disagreement rows n={_sp['n']}): "
                  f"KL(annotators||model)={_sp['kl_q_p']:.3f}, "
                  f"entropy corr={_sp['entropy_corr']:.3f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        epochs_no_improve = 0
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"✅ New best model saved (macro-F1: {best_f1:.4f})")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")
        if epochs_no_improve >= PATIENCE:
            print(f"⏹ Early stopping triggered (no improvement for {PATIENCE} epochs).")
            break

print(f"\n🎉 Training complete. Best DEV macro-F1: {best_f1:.4f}")
print(f"Best model saved at ./{best_model_path}")

# ------------------------------------------------------------------
# 8. HELD-OUT TEST  -  loaded once, reported once, never a training signal
# ------------------------------------------------------------------
_best_for_test = AutoModelForSequenceClassification.from_pretrained(best_model_path).to(device).eval()
_test_report, _test_f1 = evaluate(_best_for_test, test_loader)
print("\n================  HELD-OUT TEST (reported once)  ================")
print(f"Test macro-F1: {_test_f1:.4f}")
print(_test_report)

# ---- bootstrap CIs over test items (single seed per run: CIs quantify item
# sampling error, NOT seed variance. Do not write "robust"/"stable" from these). ----
_test_preds, _test_gold = [], []
with torch.no_grad():
    for _b in test_loader:
        _test_preds.extend(_best_for_test(input_ids=_b["input_ids"].to(device),
                           attention_mask=_b["attention_mask"].to(device)).logits.argmax(1).cpu().tolist())
        _test_gold.extend(_b["labels"].cpu().tolist() if _b["labels"].dim() == 1
                          else _b["labels"].argmax(1).cpu().tolist())
_test_preds = np.asarray(_test_preds); _test_gold = np.asarray(_test_gold)

def _boot_metric(fn, gold, pred, n_boot=2000, seed=0):
    _rng = np.random.default_rng(seed); _m = len(gold); _ix = np.arange(_m)
    _s = np.empty(n_boot)
    for _k in range(n_boot):
        _r = _rng.choice(_ix, _m, replace=True)
        _s[_k] = fn(gold[_r], pred[_r])
    return float(np.percentile(_s, 2.5)), float(np.percentile(_s, 97.5))

from sklearn.metrics import f1_score as _f1b, recall_score as _recb
_ci_lo, _ci_hi = _boot_metric(lambda g, p: _f1b(g, p, average="macro", zero_division=0),
                              _test_gold, _test_preds)
print(f"\nTest macro-F1 95% CI (2000x item bootstrap): [{_ci_lo:.4f}, {_ci_hi:.4f}]")
_cls_ci = {}
for _ci_idx, _ci_name in [(0, "hate"), (1, "offensive"), (2, "normal")]:
    _lo, _hi = _boot_metric(lambda g, p, _i=_ci_idx: _recb(g == _i, p == _i, zero_division=0),
                            _test_gold, _test_preds)
    _cls_ci[_ci_name] = (_lo, _hi)
    print(f"  recall[{_ci_name:9s}] 95% CI [{_lo:.3f}, {_hi:.3f}]")
if "record_metrics" in dir():
    record_metrics(test_macro_f1_ci_lo=_ci_lo, test_macro_f1_ci_hi=_ci_hi,
                   **{f"test_recall_{k}_ci_lo": v[0] for k, v in _cls_ci.items()},
                   **{f"test_recall_{k}_ci_hi": v[1] for k, v in _cls_ci.items()})
if "label_source" in test_df.columns:
    from sklearn.metrics import recall_score as _rec_s
    _tp = []
    with torch.no_grad():
        for _b in test_loader:
            _tp.extend(_best_for_test(input_ids=_b["input_ids"].to(device),
                       attention_mask=_b["attention_mask"].to(device)).logits.argmax(1).cpu().tolist())
    _tv = test_df.reset_index(drop=True).assign(_pred=_tp)
    # macro-F1 is meaningless when a label_source subset has only one class present
    # (ToxiGen_machine_benign / GoEmotions_positive are all-normal); report accuracy and
    # per-present-class recall instead.
    print("\nTest by label_source (accuracy + per-present-class recall; macro-F1 only when >1 class):")
    _lm3 = {0: "hate", 1: "off", 2: "normal"}
    for _src, _g in _tv.groupby("label_source"):
        _present = sorted(_g["clean_label"].unique())
        _acc = float((_g["clean_label"] == _g["_pred"]).mean())
        _recs = {_lm3[c]: float(_rec_s(_g["clean_label"] == c, _g["_pred"] == c, zero_division=0))
                 for c in _present}
        _mf1 = ("" if len(_present) < 2 else
                f" macroF1={_f1s_full(_g['clean_label'], _g['_pred']):.3f}")
        print(f"  {_src:10s} n={len(_g):6d}  acc={_acc:.3f}  recall={_recs}{_mf1}")
if USE_SOFT:
    if _spectrum_epochs:
        print("\nSpectrum check, epoch-wise (entropy_corr wanders around 0 -> negative result):")
        for _e, _ec, _kl in _spectrum_epochs:
            print(f"  epoch {_e}: entropy_corr={_ec:+.3f}  KL={_kl:.3f}")
    _spt = evaluate_disagreement(_best_for_test, test_loader)
    if _spt:
        print(f"Test spectrum: KL(annotators||model)={_spt['kl_q_p']:.3f}, "
              f"entropy corr={_spt['entropy_corr']:.3f}")
del _best_for_test
if "record_metrics" in dir():
    record_metrics(dev_macro_f1=best_f1, test_macro_f1=_test_f1,
                   n_train=len(train_df), n_dev=len(val_df), n_test=len(test_df),
                   dampen_exponent=DAMPEN_EXPONENT)
    if USE_SOFT and "_spt" in dir() and _spt:
        record_metrics(test_spectrum_kl=_spt["kl_q_p"], test_spectrum_entcorr=_spt["entropy_corr"],
                       spectrum_entcorr_by_epoch=";".join(f"{e}:{ec:+.3f}" for e, ec, _ in _spectrum_epochs))


# --- push the freshly trained best checkpoint to the HF Hub so future runs skip training ---
try:
    from huggingface_hub import create_repo
    create_repo(MODEL_REPO, exist_ok=True)
    _best = AutoModelForSequenceClassification.from_pretrained(best_model_path)
    _best.push_to_hub(MODEL_REPO)
    tokenizer.push_to_hub(MODEL_REPO)
    print(f"Pushed fine-tuned model to HF Hub: {MODEL_REPO}")
except Exception as e:
    print(f"push_to_hub failed ({e!r}) -- model is still saved locally at ./{best_model_path}")


In [ ]:
%%writefile textnorm.py
# Shared text-normalization module - imported by BOTH app.py and the
# normalization recovery-measurement cell, so the evaluated normalizer is
# exactly the deployed one. Lightweight, no retraining
# (cf. Pruthi et al. 2019; Bitton et al. 2022).
import re, functools
from spellchecker import SpellChecker

spell = SpellChecker(distance=1)   # edit-distance-1 only: ~50x faster than the
                                   # default d=2, and every single-char attack in the
                                   # recovery cell is by construction exactly one edit away.

MIN_CORRECTION_FREQ = 1e-6            # don't 'correct' toward rarer-than-this words
_TOTAL_WORDS = max(1, spell.word_frequency.total_words)
def _freq(w):
    return spell[w] / _TOTAL_WORDS

SLANG_WHITELIST = {
    "lol", "lmao", "lmfao", "omg", "wtf", "ngl", "imo", "tbh", "idk",
    "smh", "irl", "afk", "brb", "dm", "fr", "lowkey", "highkey", "bussin",
    "goated", "slay", "simp", "based", "sus", "cap", "periodt",
    "bruh", "bro", "sis", "bestie", "fam", "homie", "salty", "woke",
    "stan", "ratio", "mid", "nah", "yah", "yep", "nope", "welp", "meh",
    "ur", "u", "r", "b", "w", "n", "gonna", "wanna", "gotta", "kinda",
    "sorta", "dunno", "lemme", "gimme", "ima", "tryna", "bout", "cuz",
    "coz", "cos", "pls", "plz", "thx", "gr8", "b4", "bc", "bf", "gf",
    "tbf", "fyi", "imho", "smol", "tho", "tfw",
    "elon", "kanye", "tiktok", "insta", "reddit", "twitter", "whatsapp",
    "bert", "llm", "ai", "ml", "nlp",
}

_BENIGN_ELONGATIONS = {
    "heeey", "heyyy", "soooo", "sooo", "noooo", "yesss", "pleeeease", "hahaha", "hahah",
}

# reverse maps for the deterministic pre-spellcheck repair pass
_LEET_BACK = str.maketrans({"4": "a", "3": "e", "1": "i", "0": "o",
                            "5": "s", "7": "t", "@": "a", "$": "s"})
_HOMOGLYPH_BACK = str.maketrans({
    "\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0441": "c", "\u0440": "p",
    "\u0445": "x", "\u0443": "y", "\u0456": "i", "\u0405": "s",
})


def _dedupe_runs(w):
    """loooser -> looser -> loser: collapse runs of 3+ to 2, then to 1 if still OOV."""
    two = re.sub(r"(.)\1{2,}", r"\1\1", w)
    if not spell.unknown([two.lower()]):
        return two
    one = re.sub(r"(.)\1+", r"\1", w)
    if not spell.unknown([one.lower()]):
        return one
    return two


@functools.lru_cache(maxsize=200_000)   # perturbed corpora repeat many tokens
def _repair_token(lower, allow_spell=True):
    """High-precision deterministic repairs first; the generic spell.correction()
    fallback is gated (length, real+common target) to cut collateral edits on
    clean text."""
    for tr in (_HOMOGLYPH_BACK, _LEET_BACK):
        cand = lower.translate(tr)
        if cand != lower and not spell.unknown([cand]):
            return cand
    if re.search(r"(.)\1{2,}", lower):
        cand = _dedupe_runs(lower)
        if cand != lower and not spell.unknown([cand]):
            return cand
    stripped = re.sub(r"[.\-_*]", "", lower)
    if stripped != lower and len(stripped) > 2 and not spell.unknown([stripped]):
        return stripped
    if allow_spell and len(lower) >= 4:
        best = spell.correction(lower)
        if (best and best != lower and not spell.unknown([best])
                and _freq(best) >= MIN_CORRECTION_FREQ):
            return best
    return None


def normalize_text(text):
    words = text.split()
    out, corrections = [], []
    for i, word in enumerate(words):
        stripped = word.strip(".,!?;:\"'()-")
        lower = stripped.lower()

        if lower in SLANG_WHITELIST:
            out.append(word); continue
        if i > 0 and stripped and stripped[0].isupper() and len(stripped) > 1:
            out.append(word); continue
        if (not stripped or stripped.startswith("#") or stripped.startswith("@")
                or stripped.startswith("http") or stripped.isdigit()):
            out.append(word); continue
        if stripped.isupper() and len(stripped) > 1:
            out.append(word); continue

        _titlecase = len(stripped) > 1 and stripped[0].isupper() and stripped[1:].islower()
        cand = (_repair_token(lower, allow_spell=not _titlecase)
                if (lower and spell.unknown([lower])) else None)
        if cand:
            if stripped and stripped[0].isupper():
                cand = cand.capitalize()
            out.append(word.replace(stripped, cand))
            corrections.append((stripped, cand))
            continue
        out.append(word)

    return " ".join(out), corrections


def looks_perturbed(text):
    """Cheap check: is this text likely character-perturbed (leet, homoglyph, spacing,
    3+ char runs)? Used to apply normalize_text only when it can plausibly help - it is
    net-negative applied unconditionally (helps spell_* on HateCheck, hurts everything else)."""
    t = text.lower()
    if re.search(r"(.)\1{2,}", t):
        return True
    if any(chr(_k) in text for _k in _HOMOGLYPH_BACK):   # maketrans dict is keyed by ordinal
        return True
    if re.search(r"[a-z][0134@$]{1}[a-z]", t):            # leet digit wedged mid-word
        return True
    if re.search(r"\b(?:[a-z]\s){3,}[a-z]\b", t):        # s p a c e d out word
        return True
    return False


def has_surface_flags(text):
    for fw in re.findall(r"\b\S+\b", text.lower()):
        if re.search(r"(.)\1{2,}", fw) and fw not in _BENIGN_ELONGATIONS:
            return True, f"character-repetition misspelling: '{fw}'"

    caps_words = re.findall(r"\b[A-Z]{4,}\b", text)
    if caps_words and len(text.split()) < 20:
        return True, f"all-caps aggressive emphasis: {caps_words}"

    if re.search(r"\byou(?:'re| are) (?:a |an |such a |so )\w+", text.lower()):
        return True, "direct second-person attack structure"

    return False, ""


In [ ]:
# ============================================================
# CONFIDENCE CALIBRATION + DEFERRAL THRESHOLD
# ============================================================
# Replaces the hard-coded `confidence < 0.60` route to Gemma with two
# data-driven quantities, both fit on the stratified val split:
#   1. temperature T  - single scalar, LBFGS on val NLL (Guo et al. 2017)
#   2. route_threshold - smallest confidence cut whose KEPT set still meets
#                        TARGET_KEPT_ACCURACY, read off the risk-coverage curve
# Writes best_model/calibration.json {temperature, route_threshold, ...} which
# app.py loads, and uploads it to MODEL_REPO so cached runs inherit it.
#
# FORCE_RECALIBRATE = False re-runs even if calibration.json already exists.
# ------------------------------------------------------------
FORCE_RECALIBRATE = True
TARGET_KEPT_ACCURACY   = 0.92   # BERT must be >= this accurate on cases it does NOT defer.
                               # 0.82 sat barely above BERT-only acc so almost nothing deferred
                               # (2.7%); 0.92 lands the cut higher up the risk-coverage curve.
                               # The full curve is printed below - report it, not just one tau,
                               #  latency is reported at the selected operating point.
MAX_ROUTED_FRAC        = 0.35   # never route more than this fraction to Gemma
GEMMA_ASSUMED_ACCURACY = 0.75   # measured Gemma accuracy on deferred cases (for the "does routing help?" check)

import os, json, numpy as np, pandas as pd, torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_MDLDIR    = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
_REPO      = MODEL_REPO if "MODEL_REPO" in dir() else None
_CALIB_PATH = os.path.join(_MDLDIR, "calibration.json")
_DEV       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_DATA      = "/kaggle/working/augmented_dataset.csv"

if os.path.exists(_CALIB_PATH) and not FORCE_RECALIBRATE:
    print("calibration.json already present:")
    print(json.dumps(json.load(open(_CALIB_PATH)), indent=2))
    raise SystemExit("Calibration done (set FORCE_RECALIBRATE=True to redo).")

if not (os.path.exists(_DATA) or os.path.exists("/kaggle/working/dev_split.csv")):
    if not os.path.exists(_CALIB_PATH):
        json.dump({"temperature": 1.0, "route_threshold": 0.60,
                   "note": "default - no val data available to calibrate on"},
                  open(_CALIB_PATH, "w"), indent=2)
        print("No augmented_dataset.csv; wrote default calibration.json (T=1.0, threshold=0.60).")
    raise SystemExit("Nothing to calibrate without the val split.")

# -- DEV split: temperature and the deferral threshold are fit here, never on test --
val_df = load_eval_split("dev")
print(f"Calibrating on {len(val_df)} dev rows")

tok = AutoTokenizer.from_pretrained(_MDLDIR)
mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()


@torch.no_grad()
def _collect_logits(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                  padding=True, max_length=128).to(_DEV)
        out.append(mdl(**enc).logits.float().cpu())
    return torch.cat(out)


logits = _collect_logits(val_df["text"].tolist())
labels = torch.tensor(val_df["clean_label"].values, dtype=torch.long)


def _ece(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    acc = pred.eq(labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += (m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()).item()
    return e


pre_nll = F.cross_entropy(logits, labels).item()
pre_ece = _ece(logits.softmax(1), labels)

T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.01, max_iter=100)


def _closure():
    opt.zero_grad()
    loss = F.cross_entropy(logits / T.clamp_min(1e-3), labels)
    loss.backward()
    return loss


opt.step(_closure)
T_val = float(T.detach().clamp_min(1e-3))

post = (logits / T_val).softmax(1)
post_nll = F.cross_entropy(logits / T_val, labels).item()
post_ece = _ece(post, labels)
print(f"\nTemperature T = {T_val:.3f}")
print(f"  val NLL : {pre_nll:.4f} -> {post_nll:.4f}")
print(f"  val ECE : {pre_ece:.4f} -> {post_ece:.4f}")

# -- risk-coverage / deferral curve on CALIBRATED confidence -----------------
conf, pred = post.max(1)
conf = conf.numpy()
correct = pred.eq(labels).numpy()
rows = []
for tau in np.round(np.arange(0.34, 0.98, 0.04), 2):
    keep = conf >= tau
    cov = keep.mean()
    kept_acc = correct[keep].mean() if keep.any() else float("nan")
    exp_overall = (kept_acc * cov + GEMMA_ASSUMED_ACCURACY * (1 - cov)) if keep.any() else GEMMA_ASSUMED_ACCURACY
    rows.append((tau, cov, kept_acc, 1 - cov, exp_overall))
curve = pd.DataFrame(rows, columns=["threshold", "coverage", "kept_accuracy",
                                    "routed_frac", "expected_overall_acc"])
print("\nDeferral curve (calibrated confidence):")
print(curve.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

budget = curve[curve["routed_frac"] <= MAX_ROUTED_FRAC]
hit = budget[budget["kept_accuracy"] >= TARGET_KEPT_ACCURACY]
if len(hit):
    route_threshold = float(hit["threshold"].min())
    why = f"smallest cut with kept_acc >= {TARGET_KEPT_ACCURACY} within the {MAX_ROUTED_FRAC:.0%} routing budget"
elif len(budget):
    route_threshold = float(budget.loc[budget["expected_overall_acc"].idxmax(), "threshold"])
    why = f"no cut meets the kept-acc target inside the {MAX_ROUTED_FRAC:.0%} budget; max expected_overall_acc within budget"
else:
    route_threshold = float(curve.loc[curve["expected_overall_acc"].idxmax(), "threshold"])
    why = "routing budget infeasible; global argmax expected_overall_acc"
sel = curve[curve["threshold"] == route_threshold].iloc[0]
print(f"\nroute_threshold = {route_threshold:.2f}  ({why})")
print(f"  kept accuracy {sel.kept_accuracy:.3f} | routes {sel.routed_frac * 100:.1f}% to Gemma "
      f"| expected overall {sel.expected_overall_acc:.3f}")
if sel.expected_overall_acc <= sel.kept_accuracy + 1e-9:
    print(f"  note: at this cut routing only helps if Gemma beats {sel.kept_accuracy:.2f} on the "
          f"deferred slice (assumed {GEMMA_ASSUMED_ACCURACY}).")

calib = {
    "temperature": round(T_val, 4),
    "route_threshold": round(route_threshold, 4),
    "target_kept_accuracy": TARGET_KEPT_ACCURACY,
    "max_routed_frac": MAX_ROUTED_FRAC,
    "gemma_assumed_accuracy": GEMMA_ASSUMED_ACCURACY,
    "val_nll_pre": round(pre_nll, 4), "val_nll_post": round(post_nll, 4),
    "val_ece_pre": round(pre_ece, 4), "val_ece_post": round(post_ece, 4),
    "n_val": int(len(val_df)),
}
# one-shot sanity: what does the chosen threshold do on the untouched TEST split?
try:
    _te = load_eval_split("test")
    _tl = _collect_logits(_te["text"].tolist())
    _tp = _te["clean_label"].to_numpy()
    _tc = (_tl / T_val).softmax(1)
    _tconf = _tc.max(1).values.numpy(); _tpred = _tc.argmax(1).numpy()
    _keep = _tconf >= route_threshold
    _ka = float((_tpred[_keep] == _tp[_keep]).mean()); _rf = float(1 - _keep.mean())
    print(f"\nTEST sanity @ threshold {route_threshold:.2f}: coverage {_keep.mean():.3f} | "
          f"kept_acc {_ka:.3f} | routed {_rf:.3f}")
    calib["test_kept_accuracy_at_threshold"] = _ka
    calib["test_routed_frac_at_threshold"] = _rf
except Exception as _e:
    print("test sanity skipped:", repr(_e))

json.dump(calib, open(_CALIB_PATH, "w"), indent=2)
curve.to_csv("/kaggle/working/deferral_curve.csv", index=False)
print("\nWrote", _CALIB_PATH)
print(json.dumps(calib, indent=2))

if "record_metrics" in dir():
    record_metrics(calib_temperature=T_val, route_threshold=route_threshold,
                   dev_ece_post=post_ece,
                   test_kept_acc_at_threshold=calib.get("test_kept_accuracy_at_threshold", float("nan")),
                   test_routed_frac_at_threshold=calib.get("test_routed_frac_at_threshold", float("nan")))

if _REPO:
    try:
        from huggingface_hub import upload_file
        upload_file(path_or_fileobj=_CALIB_PATH, path_in_repo="calibration.json", repo_id=_REPO)
        print("Uploaded calibration.json ->", _REPO)
    except Exception as e:
        print("calibration.json upload skipped:", repr(e))


In [ ]:
# ============================================================
# HATECHECK  -  functional evaluation of the fine-tuned BERT
# ============================================================
# Rottger et al., ACL 2021. 29 functionalities, ~3.7k cases, each labelled
# hateful / non-hateful. Includes contrast sets built to expose false positives:
# reclaimed slurs, counter-speech that quotes hate, profanity with no target,
# abuse aimed at objects, etc. This is the fixed functional test suite that
# replaces the 12 hand-picked control sentences.
#
# The model is 3-class (hate / offensive / normal); HateCheck is binary.
# Default mapping: model "hate" -> hateful, {offensive, normal} -> non-hateful,
# because HateCheck tests HATE detection specifically (its non-hateful profanity
# and slur-reclamation cases are gold non-hateful, and would be false positives
# if "offensive" counted as hateful). Set HATE_CLASSES = {"hate", "offensive"}
# to score the looser "flagged vs not flagged" question instead.
# ------------------------------------------------------------
import os, io as _io, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

HATE_CLASSES = {"hate"}
ID2LAB = {0: "hate", 1: "offensive", 2: "normal"}
_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"


def load_hatecheck():
    """Try the official CSV, then the HF dataset, then a local Kaggle path."""
    url = "https://raw.githubusercontent.com/paul-rottger/hatecheck-data/main/test_suite_cases.csv"
    try:
        import requests
        r = requests.get(url, timeout=30); r.raise_for_status()
        return pd.read_csv(_io.StringIO(r.text))
    except Exception as e:
        print("  GitHub CSV unavailable:", repr(e))
    try:
        from datasets import load_dataset
        return load_dataset("Paul/hatecheck", split="test").to_pandas()
    except Exception as e:
        print("  HF dataset unavailable:", repr(e))
    for pth in ("/kaggle/input/hatecheck/test_suite_cases.csv",
                "/kaggle/working/test_suite_cases.csv"):
        if os.path.exists(pth):
            return pd.read_csv(pth)
    raise FileNotFoundError(
        "HateCheck not reachable. Add it as a Kaggle dataset (test_suite_cases.csv) "
        "or enable internet for this notebook.")


hc = load_hatecheck().rename(columns={"test_case": "text"})
hc = hc[["functionality", "text", "label_gold"]].dropna().reset_index(drop=True)
print(f"HateCheck: {len(hc)} cases across {hc['functionality'].nunique()} functionalities")

_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(_DEVICE).eval()


@torch.no_grad()
def _predict(texts, bs=64):
    preds = []
    for i in range(0, len(texts), bs):
        enc = _tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                   padding=True, max_length=128).to(_DEVICE)
        preds.extend(_mdl(**enc).logits.argmax(-1).cpu().tolist())
    return preds


MAPPINGS = {"hate_only": {"hate"}, "hate_or_offensive": {"hate", "offensive"}}
PRIMARY  = "hate_only"   # HateCheck tests HATE specifically; its non-hateful profanity /
                         # slur-reclamation cases become false positives if "offensive" counts.

hc["pred_idx"] = _predict(hc["text"].tolist())

try:                                             # normalized second pass (spell_* recovery)
    import sys as _sys, importlib as _il
    _sys.path.insert(0, "/kaggle/working")
    import textnorm as _tn; _il.reload(_tn)
    hc["text_norm"] = hc["text"].map(lambda t: _tn.normalize_text(t)[0])
    hc["pred_idx_norm"] = _predict(hc["text_norm"].tolist())
    _HAVE_NORM = True

    # CONDITIONAL arm: normalize ONLY when the input looks perturbed - this is what
    # app.py actually deploys (looks_perturbed() gate). Unconditional normalization
    # is net-negative on HateCheck overall, so the deployed policy must be measured
    # too, not just the unconditional one.
    _cond_gate = hc["text"].map(_tn.looks_perturbed)
    hc["gated"] = _cond_gate
    hc["text_cond"] = np.where(_cond_gate, hc["text_norm"], hc["text"])
    hc["pred_idx_cond"] = _predict(hc["text_cond"].tolist())
    _HAVE_COND = True
    print(f"conditional arm: looks_perturbed() fired on {int(_cond_gate.sum())}/{len(hc)} "
          f"cases ({_cond_gate.mean():.1%})")
except Exception as _e:
    print("normalized pass skipped:", repr(_e)); _HAVE_NORM = _HAVE_COND = False

_gh = hc["label_gold"].eq("hateful").to_numpy()

def _correct(idx_series, hate_set):
    pb = idx_series.map(lambda i: "hateful" if ID2LAB[i] in hate_set else "non-hateful")
    return (pb == hc["label_gold"]).to_numpy()

def _boot_ci(correct, n=2000, seed=0):
    rng = np.random.default_rng(seed); m = len(correct); ii = np.arange(m)
    s = np.array([correct[rng.choice(ii, m, replace=True)].mean() for _ in range(n)])
    return float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

print("\n=== HateCheck under both label mappings (raw predictions, 2000x bootstrap CI) ===")
_ms = {}
for _name, _hs in MAPPINGS.items():
    _c = _correct(hc["pred_idx"], _hs)
    _lo, _hi = _boot_ci(_c)
    _ms[_name] = dict(overall=float(_c.mean()), ci_lo=_lo, ci_hi=_hi,
                      hateful_recall=float(_c[_gh].mean()), non_hateful=float(_c[~_gh].mean()))
    _t = "  <- primary" if _name == PRIMARY else ""
    print(f"  {_name:17s}: overall {_ms[_name]['overall']:.3f}  95% CI [{_lo:.3f}, {_hi:.3f}] | "
          f"hateful-recall {_ms[_name]['hateful_recall']:.3f} | non-hateful {_ms[_name]['non_hateful']:.3f}{_t}")

HATE_CLASSES = MAPPINGS[PRIMARY]                  # primary drives the table + CSVs + downstream
hc["correct"] = _correct(hc["pred_idx"], HATE_CLASSES)
overall  = float(hc["correct"].mean())
by_label = hc.groupby("label_gold")["correct"].mean()
_agg = {"accuracy": ("correct", "mean"), "n": ("correct", "size")}
if _HAVE_NORM:
    hc["correct_norm"] = _correct(hc["pred_idx_norm"], HATE_CLASSES)
    _agg["accuracy_norm"] = ("correct_norm", "mean")
if _HAVE_COND:
    hc["correct_cond"] = _correct(hc["pred_idx_cond"], HATE_CLASSES)
    _agg["accuracy_cond"] = ("correct_cond", "mean")
by_func = (hc.groupby(["label_gold", "functionality"]).agg(**_agg).reset_index()
             .sort_values(["label_gold", "accuracy"]))

_spell_raw = _spell_norm = float("nan")
print(f"\nPrimary mapping ({PRIMARY}): overall {overall:.3f}")
if _HAVE_NORM:
    _sp = by_func[by_func["functionality"].str.startswith("spell")]
    _spell_raw, _spell_norm = float(_sp["accuracy"].mean()), float(_sp["accuracy_norm"].mean())
    print(f"  overall after normalize_text : {hc['correct_norm'].mean():.3f}   (unconditional)")
    print(f"  spell_* functionalities      : {_spell_raw:.3f} -> {_spell_norm:.3f}  (raw -> normalized)")
    if _HAVE_COND:
        _spell_cond = float(_sp["accuracy_cond"].mean())
        print(f"\n  --- THREE-ARM NORMALIZATION COMPARISON (primary mapping) ---")
        print(f"  overall  raw {overall:.3f} | unconditional {hc['correct_norm'].mean():.3f} "
              f"| conditional {hc['correct_cond'].mean():.3f}   <- deployed policy")
        print(f"  spell_*  raw {_spell_raw:.3f} | unconditional {_spell_norm:.3f} "
              f"| conditional {_spell_cond:.3f}")
        _nonsp = by_func[~by_func["functionality"].str.startswith("spell")]
        print(f"  non-spell raw {_nonsp['accuracy'].mean():.3f} "
              f"| unconditional {_nonsp['accuracy_norm'].mean():.3f} "
              f"| conditional {_nonsp['accuracy_cond'].mean():.3f}"
              "   <- collateral damage the gate is meant to avoid")

    # per-functionality (F25-F29), not averaged, for direct comparison to Rottger et al.
    # 2021 Table 4 (e.g. 43.9% spell_space_add_h, 48.0% spell_leet_h) and Bitton et al. 2022.
    _spell_tab = _sp.copy()
    _spell_tab["recovery_pp"] = _spell_tab["accuracy_norm"] - _spell_tab["accuracy"]
    print("\nSpelling functionalities (per functionality, not averaged):")
    with pd.option_context("display.width", 140):
        print(_spell_tab[["functionality", "n", "accuracy", "accuracy_norm", "recovery_pp"]]
              .to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    _spell_tab.to_csv("/kaggle/working/hatecheck_spelling_table.csv", index=False)

print("\nPer functionality (worst first within each gold label):")
with pd.option_context("display.max_rows", None, "display.width", 140):
    print(by_func.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

hc.to_csv("/kaggle/working/hatecheck_results.csv", index=False)
by_func.to_csv("/kaggle/working/hatecheck_by_functionality.csv", index=False)
pd.DataFrame(_ms).T.to_csv("/kaggle/working/hatecheck_mapping_comparison.csv")

# CONSTRUCT DIVERGENCE: HateCheck gold is "hate targeting a protected group". The deployment
# construct (cyberbullying) ALSO flags individual/object-directed abuse. On functionalities
# like target_indiv_nh / target_obj_nh ("You make me sick", "I will hurt you"; gold =
# non-hateful), a correct cyberbullying flag is scored as a HateCheck error -> optimising
# HateCheck accuracy actively pushes the model away from the deployment task here.
_INDIV_ABUSE_NH = {"target_indiv_nh", "target_obj_nh"}
_ia = hc[hc["functionality"].isin(_INDIV_ABUSE_NH)]
if len(_ia):
    _flag_rate = float((_ia["pred_idx"].map(lambda i: ID2LAB[i]) != "normal").mean())
    print(f"\nConstruct divergence - individual/object-directed abuse (n={len(_ia)}, HateCheck "
          f"gold=non-hateful): model flags {_flag_rate:.3f}. For cyberbullying these are true "
          f"positives; HateCheck counts them as errors.")
    if "record_metrics" in dir():
        record_metrics(hatecheck_indiv_abuse_flag_rate=_flag_rate, hatecheck_indiv_abuse_n=len(_ia))
if "record_metrics" in dir():
    record_metrics(
        hatecheck_overall_hateonly=_ms["hate_only"]["overall"],
        hatecheck_overall_hate_or_off=_ms["hate_or_offensive"]["overall"],
        hatecheck_ci_lo=_ms[PRIMARY]["ci_lo"], hatecheck_ci_hi=_ms[PRIMARY]["ci_hi"],
        hatecheck_hateful_recall=_ms[PRIMARY]["hateful_recall"],
        hatecheck_nonhateful=_ms[PRIMARY]["non_hateful"],
        hatecheck_overall_norm=(float(hc["correct_norm"].mean()) if _HAVE_NORM else float("nan")),
        hatecheck_spell_raw=_spell_raw, hatecheck_spell_norm=_spell_norm)
print("\nSaved -> hatecheck_results.csv, hatecheck_by_functionality.csv, "
      "hatecheck_mapping_comparison.csv"
      + (", hatecheck_spelling_table.csv" if _HAVE_NORM else ""))


In [ ]:
# ============================================================
# FAILURE ANALYSIS  -  where the fine-tuned BERT breaks on informal text
# ============================================================
# Buckets misclassifications into reproducible categories and attaches
# occlusion-based token attributions as evidence. Sources, in order:
#   (1) inline probe set  - always runs, crafted one-per-category
#   (2) hatecheck_results.csv        - if the HateCheck cell has run
#   (3) stratified val split of augmented_dataset.csv - if present
# Output: failure_analysis_cases.csv  +  a printed bucket summary.
#
# Buckets are heuristic and inspectable (rule stated below, text kept in the
# CSV so every assignment can be audited by hand):
#   political_economic_vocab  pred=hate, gold!=hate, political/economic/country term present
#   oov_or_rare_token         pred=hate, gold!=hate, a word splits into >=3 wordpieces or [UNK]
#   negvalence_no_target      pred=hate, gold!=hate, negative-valence word, no 2nd/3rd-person target
#   informal_register_missed  pred=normal, gold in {hate,offensive}, 2nd-person + informal markers
#   hate_downgraded           pred=offensive, gold=hate
#   offensive_upgraded        pred=hate, gold=offensive
# ------------------------------------------------------------
import os, re, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ID2LAB   = {0: "hate", 1: "offensive", 2: "normal"}
_DEV     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_MDLDIR  = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
TOKEN_CAP_PER_BUCKET = 60          # occlusion attributions computed for this many per bucket

POLI = {
    "liberal", "liberals", "conservative", "conservatives", "capitalist", "capitalism",
    "socialist", "socialism", "communist", "communism", "marxist", "marxism", "nationalist",
    "fascist", "fascism", "democrat", "democrats", "republican", "republicans", "tory",
    "tories", "labour", "leftist", "rightwing", "left-wing", "right-wing",
    "immigration", "immigrant", "immigrants", "migrant", "refugee", "economy", "economic",
    "gdp", "tax", "taxes", "taxation", "policy", "parliament", "congress", "senate",
    "election", "elections", "government", "regime", "coalition", "referendum", "brexit",
    "sanctions", "tariff", "tariffs", "welfare", "deregulation",
    "india", "indian", "china", "chinese", "america", "american", "russia", "russian",
    "israel", "israeli", "palestine", "palestinian", "pakistan", "pakistani", "britain",
    "british", "france", "french", "germany", "german", "europe", "european", "africa",
    "african", "mexico", "mexican", "ukraine", "ukrainian", "iran", "iranian",
}
NEGVAL = {
    "hate", "miserable", "pointless", "hopeless", "worthless", "useless", "tired",
    "exhausted", "depressed", "sad", "done", "worst", "awful", "terrible", "stuck",
    "failing", "fail", "failed", "cry", "crying", "empty", "numb", "drained", "broken",
}
INFORMAL_SLANG = {
    "u", "ur", "yr", "lmao", "lmaoo", "lmfao", "lol", "lolol", "omg", "bruh", "bro",
    "idk", "tbh", "ngl", "fr", "af", "lil", "smh", "istg", "dawg",
}

# -- inline probe set: (text, expected_label, category_being_probed) --------------
PROBE = [
    ("eish, this traffic is unbelievable today",                     "normal",    "oov_or_rare_token"),
    ("wallahi I forgot my keys again",                               "normal",    "oov_or_rare_token"),
    ("load-shedding schedule changed again, yoh",                    "normal",    "oov_or_rare_token"),
    ("sana all, the meeting moved to 3pm",                           "normal",    "oov_or_rare_token"),
    ("chalo, let us grab some chai before the standup",              "normal",    "oov_or_rare_token"),
    ("India is a mixed economy with public and private sectors",     "normal",    "political_economic_vocab"),
    ("the liberal and conservative parties disagree on tax policy",  "normal",    "political_economic_vocab"),
    ("capitalism and socialism balance differently across Europe",   "normal",    "political_economic_vocab"),
    ("immigration reform has been debated in Congress for decades",  "normal",    "political_economic_vocab"),
    ("the nationalist coalition lost seats in the last election",    "normal",    "political_economic_vocab"),
    ("I hate mondays so much, this week is already the worst",       "normal",    "negvalence_no_target"),
    ("this whole situation is miserable and feels pointless",        "normal",    "negvalence_no_target"),
    ("I feel completely drained and nothing I do seems to work",     "normal",    "negvalence_no_target"),
    ("the build keeps failing and I am exhausted",                   "normal",    "negvalence_no_target"),
    ("honestly the worst, everything is broken today",              "normal",    "negvalence_no_target"),
    ("hey i think u are a fucking loooser",                          "offensive", "informal_register_missed"),
    ("ur so dumb it is actually impressive lmaooo",                  "offensive", "informal_register_missed"),
    ("nobody likes u, just leave the group chat already",           "offensive", "informal_register_missed"),
    ("wow ur really the stupidest person here huh",                  "offensive", "informal_register_missed"),
    ("k well u clearly have zero brain cells",                       "offensive", "informal_register_missed"),
    ("You did a great job on this project.",                         "normal",    "control_correct"),
    ("Thanks so much for helping me move today!",                    "normal",    "control_correct"),
    ("Some groups just aren't built for this kind of work.",         "hate",      "control_correct"),
    ("Go back to where you came from, you don't belong here.",       "hate",      "control_correct"),
]

_tok = AutoTokenizer.from_pretrained(_MDLDIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()


@torch.no_grad()
def _predict3(texts, bs=64):
    labs, confs = [], []
    for i in range(0, len(texts), bs):
        enc = _tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                   padding=True, max_length=128).to(_DEV)
        pr = _mdl(**enc).logits.softmax(-1).cpu().numpy()
        labs  += [ID2LAB[int(r.argmax())] for r in pr]
        confs += [float(r.max()) for r in pr]
    return labs, confs


@torch.no_grad()
def _top_tokens(text, n=3):
    words = text.split()
    if len(words) < 2:
        return ""
    variants = [text] + [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))]
    enc = _tok(variants, return_tensors="pt", truncation=True, padding=True,
               max_length=128).to(_DEV)
    pr = _mdl(**enc).logits.softmax(-1)
    k = int(pr[0].argmax()); base = float(pr[0, k])
    deltas = sorted(((words[i], base - float(pr[i + 1, k])) for i in range(len(words))),
                    key=lambda x: x[1], reverse=True)
    return "; ".join(f"{w}({d:+.3f})" for w, d in deltas[:n] if d > 0)


def _oov_words(text):
    out = []
    for w in re.findall(r"[A-Za-z][A-Za-z'\-]{1,}", text):
        pieces = _tok.tokenize(w)
        if pieces and (_tok.unk_token in pieces or len(pieces) >= 3):
            out.append(w)
    return out


def _informal(text):
    t = text.lower()
    if re.search(r"(.)\1{2,}", t):
        return True
    if INFORMAL_SLANG & set(re.findall(r"[a-z']+", t)):
        return True
    letters = re.sub(r"[^A-Za-z]", "", text)
    return bool(letters) and text == t and len(text.split()) > 4      # all-lowercase running text


def _bucket(text, gold, pred):
    t = text.lower()
    toks = set(re.findall(r"[a-z'\-]{2,}", t))
    target = re.search(r"\b(you|u|ur|yr|your|youre|y'all|yall|he|she|they|them|him|her)\b", t)
    if pred == "hate" and gold != "hate":
        if toks & POLI:
            return "political_economic_vocab"
        if _oov_words(text):
            return "oov_or_rare_token"
        if not target and (toks & NEGVAL):
            return "negvalence_no_target"
        return "other_false_positive"
    if pred == "normal" and gold in ("hate", "offensive"):
        if target and _informal(text):
            return "informal_register_missed"
        return "other_false_negative"
    if (pred, gold) == ("offensive", "hate"):
        return "hate_downgraded"
    if (pred, gold) == ("hate", "offensive"):
        return "offensive_upgraded"
    return "other"


# -- assemble sources -----------------------------------------------------------
frames = [pd.DataFrame({"source": "probe",
                        "functionality": [c for _, _, c in PROBE],
                        "text": [x for x, _, _ in PROBE],
                        "gold": [g for _, g, _ in PROBE]})]

if os.path.exists("/kaggle/working/hatecheck_results.csv"):
    hc = pd.read_csv("/kaggle/working/hatecheck_results.csv")
    frames.append(pd.DataFrame({"source": "hatecheck",
                                "functionality": hc["functionality"],
                                "text": hc["text"],
                                "gold": np.where(hc["label_gold"].eq("hateful"), "hate", "normal")}))
else:
    print("hatecheck_results.csv not found - run the HateCheck cell to include it.")

try:
    _te = load_eval_split("test")
    frames.append(pd.DataFrame({"source": "test",
                                "functionality": "",
                                "text": _te["text"].astype(str),
                                "gold": _te["clean_label"].map(ID2LAB)}))
    print(f"Held-out TEST rows for failure mining: {len(_te)}")
except Exception as _e:
    print("held-out test split unavailable:", repr(_e))

cases = pd.concat(frames, ignore_index=True)
cases["pred"], cases["confidence"] = _predict3(cases["text"].tolist())
cases["misclassified"] = cases["pred"] != cases["gold"]
cases["bucket"] = [_bucket(t, g, p) for t, g, p in
                   zip(cases["text"], cases["gold"], cases["pred"])]

fails = cases[cases["misclassified"]].copy()
fails["top_tokens"] = ""
for b, grp in fails.groupby("bucket"):
    for idx in grp.index[:TOKEN_CAP_PER_BUCKET]:
        fails.at[idx, "top_tokens"] = _top_tokens(fails.at[idx, "text"])
cases["top_tokens"] = ""
cases.loc[fails.index, "top_tokens"] = fails["top_tokens"]

# -- HEADLINE numbers: NATURAL failures only (hatecheck + held-out test). The PROBE
#    set is crafted in exactly the categories _bucket() detects, so its bucket
#    counts are true by construction -> reported separately as illustration.
NATURAL_SOURCES = {"hatecheck", "test"}
# non-hateful HateCheck cases where the model calling "offensive" (not "normal") is a
# defensible read, not a bucket-worthy FP: general profanity / abuse with no protected-
# identity target. Excluded from the natural bucket counts (kept in cases.csv).
HATECHECK_AMBIGUOUS_NH = {"profanity_nh", "target_indiv_nh", "target_obj_nh"}
nat = cases[cases["source"].isin(NATURAL_SOURCES)
            & ~((cases["source"] == "hatecheck")
                & cases["functionality"].isin(HATECHECK_AMBIGUOUS_NH))]
nat_fail = nat[nat["misclassified"]].copy()

summ = (nat_fail.groupby("bucket")
             .agg(n=("text", "size"), mean_conf=("confidence", "mean"), example=("text", "first"))
             .sort_values("n", ascending=False))
print(f"\n=== NATURAL failures ({len(nat)} cases from {sorted(NATURAL_SOURCES)} | "
      f"{len(nat_fail)} misclassified = {len(nat_fail) / max(1, len(nat)) * 100:.1f}%) ===")
with pd.option_context("display.max_colwidth", 70, "display.width", 150):
    print(summ.to_string(float_format=lambda x: f"{x:.3f}"))

for key, msg in [("political_economic_vocab", "Over-flags political / economic speech"),
                 ("informal_register_missed", "Under-flags informal-register abuse")]:
    if key in summ.index:
        r = summ.loc[key]
        print(f"\n{msg}: {int(r.n)} natural cases, mean confidence {r.mean_conf:.2f}")

print("\nSample NATURAL failures per bucket:")
for b in summ.index:
    print(f"\n[{b}]")
    for _, r in nat_fail[nat_fail.bucket == b].head(3).iterrows():
        tt = f"   tokens: {r.top_tokens}" if r.top_tokens else ""
        print(f"  gold={r.gold:9s} pred={r.pred:9s} conf={r.confidence:.2f} | {r.text[:90]}{tt}")

probe = cases[cases["source"] == "probe"].copy()
if len(probe):
    print(f"\n=== CRAFTED PROBE SET ({len(probe)} sentences, "
          f"{int(probe['misclassified'].sum())} misclassified) - illustrative, NOT a rate ===")
    with pd.option_context("display.max_colwidth", 58, "display.width", 160):
        print(probe[["functionality", "gold", "pred", "confidence", "misclassified", "text"]]
              .to_string(index=False))
    probe.to_csv("/kaggle/working/failure_analysis_probes.csv", index=False)
    print("\n  probe buckets (crafted; RESOLVED = model now handles it, LIVE = still fails):")
    for _bk, _pg in probe.groupby("functionality"):
        _nfail = int(_pg["misclassified"].sum())
        _tag = "RESOLVED" if _nfail == 0 else f"LIVE ({_nfail}/{len(_pg)} fail)"
        print(f"    {_bk:26s} {_tag}")

cases.to_csv("/kaggle/working/failure_analysis_cases.csv", index=False)
summ.to_csv("/kaggle/working/failure_analysis_summary.csv")
if "record_metrics" in dir():
    record_metrics(
        failure_natural_misclass_rate=len(nat_fail) / max(1, len(nat)),
        failure_political_fp=int(summ.loc["political_economic_vocab", "n"]) if "political_economic_vocab" in summ.index else 0,
        failure_informal_fn=int(summ.loc["informal_register_missed", "n"]) if "informal_register_missed" in summ.index else 0)
print("\nSaved -> failure_analysis_cases.csv (all), failure_analysis_summary.csv (natural), "
      "failure_analysis_probes.csv (crafted)")


In [ ]:
# ============================================================
# RATIONALE OVERLAP  -  do the model's occlusion attributions land on the
# tokens human annotators marked as the reason?  (HateXplain "plausibility")
# ============================================================
# HateXplain ships token-level rationale masks. Mathew et al. 2021 (AAAI) evaluate
# plausibility as token-F1 / AUPRC between model importance and human rationale.
# Here: leave-one-out occlusion (same as get_word_importances in app.py) on the
# held-out TEST rows that came from HateXplain, vs the majority-annotator mask.
# NOTE: matching is by exact " ".join(post_tokens) == text, which holds only on the
# SOFT_LABEL_TRAINING path (hateXplain text is built that way there).
# ------------------------------------------------------------
RATIONALE_TOPK     = None   # None -> k = |rationale| per example; or an int
RATIONALE_MAX_ROWS = 400    # occlusion is O(tokens) fwd passes/row; cap it

import os, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import average_precision_score

_MDLDIR = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _load_hatexplain_rationales():
    try:
        from datasets import load_dataset
        for _p in ("hatexplain", "Paul/hatexplain"):
            try:
                _ds = load_dataset(_p, trust_remote_code=True)
                return [ex for sp in _ds for ex in _ds[sp]]
            except Exception:
                pass
    except Exception:
        pass
    import requests
    _u = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json"
    _raw = requests.get(_u, timeout=60).json()
    return [{"post_tokens": r["post_tokens"], "rationales": r.get("rationales", [])}
            for r in _raw.values()]


_hx = _load_hatexplain_rationales()
_rat = {}
for _ex in _hx:
    _toks = _ex["post_tokens"]
    _rs = [r for r in (_ex.get("rationales") or []) if isinstance(r, list) and len(r) == len(_toks)]
    if not _rs:
        continue
    _need = len(_rs) // 2 + 1
    _m = (np.array(_rs).sum(0) >= _need).astype(int)   # majority of annotators who supplied a rationale
    if _m.sum():
        _rat[" ".join(_toks)] = (_toks, _m)
print(f"HateXplain posts with a non-empty majority rationale: {len(_rat)}")

_te = load_eval_split("test")
if "source" in _te.columns:
    _te = _te[_te["source"].astype(str) == "hateXplain"]
_pairs = [(t, _rat[t]) for t in _te["text"].astype(str).tolist() if t in _rat]
if RATIONALE_MAX_ROWS:
    _pairs = _pairs[:RATIONALE_MAX_ROWS]
print(f"TEST rows matched to a rationale: {len(_pairs)}")
if not _pairs:
    print("No overlap - is SOFT_LABEL_TRAINING on, and are hateXplain rows in the test split?")
else:
    _tok = AutoTokenizer.from_pretrained(_MDLDIR)
    _mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()

    @torch.no_grad()
    def _occ(words):
        variants = [" ".join(words)] + [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))]
        enc = _tok(variants, return_tensors="pt", truncation=True, padding=True, max_length=128).to(_DEV)
        p = _mdl(**enc).logits.softmax(-1)
        k = int(p[0].argmax()); base = float(p[0, k])
        return np.array([base - float(p[i + 1, k]) for i in range(len(words))])

    _f1, _iou, _ap = [], [], []
    for _text, (_toks, _mask) in _pairs:
        if len(_toks) < 2:
            continue
        _s = _occ(_toks)
        _k = int(_mask.sum()) if RATIONALE_TOPK is None else RATIONALE_TOPK
        _k = max(1, min(_k, len(_toks)))
        _top = np.zeros(len(_toks), int)
        _top[np.argsort(-_s)[:_k]] = 1
        _tp = int((_top & _mask).sum())
        _fp = int((_top & (1 - _mask)).sum())
        _fn = int(((1 - _top) & _mask).sum())
        _p = _tp / (_tp + _fp) if _tp + _fp else 0.0
        _r = _tp / (_tp + _fn) if _tp + _fn else 0.0
        _f1.append(0.0 if _p + _r == 0 else 2 * _p * _r / (_p + _r))
        _iou.append(_tp / (_tp + _fp + _fn) if _tp + _fp + _fn else 0.0)
        if 0 < _mask.sum() < len(_mask):
            try:
                _ap.append(average_precision_score(_mask, _s))
            except Exception:
                pass

    print(f"\n=== Rationale plausibility on {len(_f1)} HateXplain TEST rows ===")
    print(f"  token-F1 (top-k vs majority rationale) : {np.mean(_f1):.3f}")
    print(f"  token-IoU                              : {np.mean(_iou):.3f}")
    print(f"  AUPRC (attribution vs rationale)       : "
          + (f"{np.mean(_ap):.3f}" if _ap else "n/a"))
    print("  (Mathew et al. 2021 report token-F1 ~0.5 / AUPRC ~0.85 for a rationale-SUPERVISED")
    print("   model; ours is post-hoc occlusion with no rationale supervision, so lower is expected.)")
    pd.DataFrame({"token_f1": _f1, "token_iou": _iou}).to_csv(
        "/kaggle/working/rationale_overlap.csv", index=False)
    if "record_metrics" in dir():
        record_metrics(rationale_token_f1=float(np.mean(_f1)), rationale_token_iou=float(np.mean(_iou)),
                       rationale_auprc=(float(np.mean(_ap)) if _ap else float("nan")),
                       rationale_n=len(_f1))
    print("\nSaved -> rationale_overlap.csv")


In [ ]:
# ============================================================
# NORMALIZATION RECOVERY  -  how much of a character-level attack does
# normalize_text() undo, per attack type
# ============================================================
# For each attack, accuracy on: (a) clean, (b) perturbed, (c) perturbed
# then normalize_text().   recovery_frac = (c - b) / (a - b).
# Also: the normalizer's edit count on attacked text, and its collateral
# edit rate on clean text. Attacks = Pruthi et al. 2019 (swap/drop/insert/
# keyboard) + the hate-speech-specific ones (elongate/leet/homoglyph/spaced).
# ------------------------------------------------------------
import os, re, sys, random, importlib, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.insert(0, "/kaggle/working")
import textnorm; importlib.reload(textnorm)
from textnorm import normalize_text, SLANG_WHITELIST

SEED = 42
N_EVAL = 600           # sampled from the val split (raise for tighter CIs, lower for speed)
WORDS_PER_SENT = 2     # eligible content words each attack perturbs per sentence
_MDLDIR = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FALLBACK_EVAL = [
    ("you are a complete idiot and everyone knows it", 1),
    ("go back to where you came from you do not belong here", 0),
    ("what a lovely afternoon for a walk in the park", 2),
    ("thanks so much for helping me move today", 2),
    ("some groups just are not built for this kind of work", 0),
    ("honestly the traffic this morning was unbelievable", 2),
    ("she is the smartest person on the whole team", 2),
    ("nobody wants you here so just leave already", 1),
    ("the meeting has been moved to three o'clock", 2),
    ("these people are animals and should be caged", 0),
    ("i really appreciate everything you did for us", 2),
    ("you are such a pathetic waste of space", 1),
]

try:
    tdf = load_eval_split("test")
    tdf = (tdf.groupby("clean_label", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), max(1, N_EVAL // 3)), random_state=SEED)))
    eval_texts = tdf["text"].astype(str).tolist()
    eval_labels = tdf["clean_label"].tolist()
    print(f"Eval set: {len(eval_texts)} rows from the held-out TEST split")
except Exception as _e:
    print(f"test split unavailable ({_e!r}) - using the {len(FALLBACK_EVAL)}-row fallback set")
    eval_texts = [t for t, _ in FALLBACK_EVAL]
    eval_labels = [y for _, y in FALLBACK_EVAL]

_tok = AutoTokenizer.from_pretrained(_MDLDIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()


@torch.no_grad()
def _acc(texts, labels, bs=64):
    ok = 0
    for i in range(0, len(texts), bs):
        enc = _tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                   padding=True, max_length=128).to(_DEV)
        pr = _mdl(**enc).logits.argmax(-1).cpu().tolist()
        ok += sum(int(p == y) for p, y in zip(pr, labels[i:i + bs]))
    return ok / len(texts)


_QWERTY = {"q": "wa", "w": "qe", "e": "wr", "r": "et", "t": "ry", "y": "tu", "u": "yi",
           "i": "uo", "o": "ip", "p": "ol", "a": "sqz", "s": "adw", "d": "sfe", "f": "dgr",
           "g": "fht", "h": "gjy", "j": "hku", "k": "jli", "l": "kop", "z": "xas", "x": "zcd",
           "c": "xvf", "v": "cbg", "b": "vnh", "n": "bmj", "m": "nk"}
_LEET = {"a": "4", "e": "3", "i": "1", "o": "0", "s": "5", "t": "7"}
_HOMO = {"a": "\u0430", "e": "\u0435", "o": "\u043e", "c": "\u0441", "p": "\u0440",
         "x": "\u0445", "y": "\u0443"}


def _swap(w, rng):
    if len(w) < 4: return w
    i = rng.randint(1, len(w) - 3)
    return w[:i] + w[i + 1] + w[i] + w[i + 2:]

def _drop(w, rng):
    if len(w) < 4: return w
    i = rng.randint(1, len(w) - 2)
    return w[:i] + w[i + 1:]

def _insert(w, rng):
    if len(w) < 3: return w
    i = rng.randint(1, len(w) - 1)
    return w[:i] + rng.choice("abcdefghijklmnopqrstuvwxyz") + w[i:]

def _keyboard(w, rng):
    if len(w) < 3: return w
    i = rng.randint(1, len(w) - 1); c = w[i].lower()
    return w[:i] + (rng.choice(_QWERTY[c]) if c in _QWERTY else w[i]) + w[i + 1:]

def _elongate(w, rng):
    if len(w) < 3: return w
    i = rng.randint(1, len(w) - 1)
    return w[:i] + w[i] * rng.randint(3, 5) + w[i + 1:]

def _leet(w, rng):
    return "".join(_LEET.get(c.lower(), c) for c in w)

def _homoglyph(w, rng):
    return "".join(_HOMO.get(c.lower(), c) for c in w)

def _spaced(w, rng):
    return ".".join(w)


ATTACKS = {"swap": _swap, "drop": _drop, "insert": _insert, "keyboard": _keyboard,
           "elongate": _elongate, "leet": _leet, "homoglyph": _homoglyph, "spaced": _spaced}
_WORD = re.compile(r"[A-Za-z]+")


def _perturb(text, fn, rng):
    toks = text.split()
    idxs = [j for j, t in enumerate(toks)
            if _WORD.fullmatch(t) and len(t) >= 4 and t.lower() not in SLANG_WHITELIST]
    rng.shuffle(idxs)
    n = 0
    for j in idxs[:WORDS_PER_SENT]:
        new = fn(toks[j], rng)
        if new != toks[j]:
            toks[j] = new; n += 1
    return " ".join(toks), n


clean_acc = _acc(eval_texts, eval_labels)
clean_edits = sum(len(normalize_text(t)[1]) for t in eval_texts)
print(f"\nClean accuracy: {clean_acc:.3f}")
print(f"Normalizer collateral on clean text: {clean_edits} edits / {len(eval_texts)} sents "
      f"({clean_edits / len(eval_texts):.3f} per sent)\n")

rows = []
for ai, (name, fn) in enumerate(ATTACKS.items()):
    print(f"  [{ai + 1}/{len(ATTACKS)}] {name} ...", flush=True)
    rng = random.Random(SEED + ai)
    pert, tok_hits = [], 0
    for t in eval_texts:
        p, k = _perturb(t, fn, rng)
        pert.append(p); tok_hits += k
    res = [normalize_text(p) for p in pert]
    norm = [r[0] for r in res]
    edits = sum(len(r[1]) for r in res)
    pert_acc = _acc(pert, eval_labels)
    norm_acc = _acc(norm, eval_labels)
    denom = clean_acc - pert_acc
    rec = (norm_acc - pert_acc) / denom if denom > 1e-6 else float("nan")
    rows.append((name, clean_acc, pert_acc, norm_acc, norm_acc - pert_acc, rec, tok_hits, edits))

summary = pd.DataFrame(rows, columns=["attack", "clean_acc", "perturbed_acc", "normalized_acc",
                                      "abs_gain", "recovery_frac", "tokens_perturbed", "normalizer_edits"])
print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
summary.to_csv("/kaggle/working/normalization_recovery.csv", index=False)

_valid = summary["recovery_frac"].dropna()
if len(_valid):
    print(f"\nMean recovery across attacks: {_valid.mean():.2f}")
    print(f"Best : {summary.loc[summary.recovery_frac.idxmax(), 'attack']} "
          f"({summary.recovery_frac.max():.2f})")
    print(f"Worst: {summary.loc[summary.recovery_frac.idxmin(), 'attack']} "
          f"({summary.recovery_frac.min():.2f})")
if "record_metrics" in dir():
    record_metrics(norm_recovery_mean=(float(_valid.mean()) if len(_valid) else float("nan")),
                   norm_collateral_per_sent=clean_edits / max(1, len(eval_texts)))
print("\nSaved -> normalization_recovery.csv")
print("NOTE: compare this table side-by-side with hatecheck_spelling_table.csv - synthetic "
      "perturbation benchmarks overstate normalizer value vs a human-crafted functional suite "
      "(the NoisyHate argument).")


In [ ]:
# ============================================================
# SECOND-CORPUS EVALUATION  -  cross-domain generalization (inference only)
# ============================================================
# Muminovic's YouTube cyberbullying dataset (English subset) ships alongside a
# published GPT-4.1 / Gemini / Claude baseline, so this is the one eval in the
# notebook with an external LLM comparison point already published for it.
# No training happens here.
#
# NOTE: I could not verify the exact HF dataset id / column schema for this
# corpus from this environment. Fill in SECOND_CORPUS_SOURCE (a HF dataset id)
# or drop a CSV as a Kaggle input matching SECOND_CORPUS_CSV_GLOB, then re-run.
# The loader prints the columns it finds and fails with instructions rather
# than guessing - a silent wrong-column read would be worse than no result.
# ------------------------------------------------------------
SECOND_CORPUS_SOURCE   = ""   # <-- Hugging Face dataset id, e.g. "user/yt-cyberbullying-en"
# Muminovic corpus: Kaggle dataset alinashifa/muminovic-dataset, file comments.csv.
# Recursive glob so it is found wherever Kaggle mounts it (NOT /kaggle/input/datasets/...).
SECOND_CORPUS_CSV_GLOB = "/kaggle/input/**/comments.csv"
SECOND_CORPUS_TEXT_COLS  = ["comment_text", "text", "comment", "Text", "content"]
SECOND_CORPUS_LABEL_COLS = ["human_label", "label", "Label", "cyberbullying", "is_bullying",
                            "class", "Class", "cyberbullying_type", "CB_Label"]
# 1 = bullying-adjacent label strings; extend with the corpus's own values if needed.
SECOND_CORPUS_LLM_COLS   = ["gpt_pred", "gemini_pred", "claude_pred", "gpt4_pred", "gpt-4.1_pred",
                            "gpt_4_1_pred", "gemini_prediction", "claude_prediction"]
SECOND_CORPUS_MAX_ROWS = 3000   # cap for inference time; set None to run all

import os, glob as _sglob, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, f1_score

_MDLDIR = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ID2LAB3 = {0: "hate", 1: "offensive", 2: "normal"}
BULLY_CLASSES = {"hate", "offensive"}   # our verdicts that count as "bullying" for this comparison


def _load_second_corpus():
    if SECOND_CORPUS_SOURCE:
        try:
            from datasets import load_dataset
            ds = load_dataset(SECOND_CORPUS_SOURCE)
            split = "test" if "test" in ds else ("train" if "train" in ds else list(ds.keys())[0])
            return ds[split].to_pandas(), f"HF:{SECOND_CORPUS_SOURCE}[{split}]"
        except Exception as e:
            print(f"  HF load of '{SECOND_CORPUS_SOURCE}' failed: {e!r}")
    hits = _sglob.glob(SECOND_CORPUS_CSV_GLOB, recursive=True)
    _lit = "/kaggle/input/datasets/alinashifa/muminovic-dataset/comments.csv"
    for _h in hits + ([_lit] if os.path.exists(_lit) else []):
        return pd.read_csv(_h), _h
    return None, None


_df2, _src2 = _load_second_corpus()
if _df2 is None:
    print("Second-corpus dataset not found. Either:\n"
          "  1) add it as a Kaggle input dataset (a CSV matching "
          f"'{SECOND_CORPUS_CSV_GLOB}'), or\n"
          "  2) set SECOND_CORPUS_SOURCE to its Hugging Face dataset id,\n"
          "then re-run this cell. Inference only - no training needed either way.")
    raise SystemExit
print(f"Loaded second corpus from {_src2}: {len(_df2)} rows")
print(f"  columns: {list(_df2.columns)}")

_text_col = next((c for c in SECOND_CORPUS_TEXT_COLS if c in _df2.columns), None)
_label_col = next((c for c in SECOND_CORPUS_LABEL_COLS if c in _df2.columns), None)
if _text_col is None or _label_col is None:
    print(f"Could not auto-detect text/label columns in {list(_df2.columns)}.\n"
          "Add the real names to SECOND_CORPUS_TEXT_COLS / SECOND_CORPUS_LABEL_COLS above and re-run.")
    raise SystemExit
print(f"  using text='{_text_col}', label='{_label_col}'")


def _to_binary_bully(v):
    s = str(v).strip().lower()
    if s in ("1", "true", "yes", "bully", "bullying", "cyberbullying", "toxic",
             "offensive", "hate", "hateful", "abusive", "harassment"):
        return 1
    if s in ("0", "false", "no", "not_bullying", "not bullying", "non_bullying",
             "non-bullying", "normal", "none", "not cyberbullying", "clean", "neutral"):
        return 0
    try:
        return int(float(s) > 0)
    except ValueError:
        return np.nan


_df2["_gold_bin"] = _df2[_label_col].map(_to_binary_bully)
_dropped = int(_df2["_gold_bin"].isna().sum())
_df2 = _df2.dropna(subset=["_gold_bin", _text_col]).reset_index(drop=True)
_df2["_gold_bin"] = _df2["_gold_bin"].astype(int)
if _dropped:
    print(f"  dropped {_dropped} rows with an unrecognised label value - check the mapping above")
if SECOND_CORPUS_MAX_ROWS and len(_df2) > SECOND_CORPUS_MAX_ROWS:
    _df2 = (_df2.groupby("_gold_bin", group_keys=False)
                .apply(lambda g: g.sample(min(len(g), SECOND_CORPUS_MAX_ROWS // 2), random_state=42)))
    print(f"  sampled down to {len(_df2)} rows ({SECOND_CORPUS_MAX_ROWS} cap)")

_tok = AutoTokenizer.from_pretrained(_MDLDIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()


@torch.no_grad()
def _predict3(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = _tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                   padding=True, max_length=128).to(_DEV)
        out.extend(_mdl(**enc).logits.argmax(-1).cpu().tolist())
    return out


_df2["pred_idx"] = _predict3(_df2[_text_col].astype(str).tolist())
_df2["pred_bin"] = _df2["pred_idx"].map(lambda i: 1 if ID2LAB3[i] in BULLY_CLASSES else 0)

_gold = _df2["_gold_bin"].to_numpy()
_pred = _df2["pred_bin"].to_numpy()
_acc = float((_gold == _pred).mean())
_prec, _rec, _f1, _ = precision_recall_fscore_support(_gold, _pred, average="binary", zero_division=0)
_macro_f1 = f1_score(_gold, _pred, average="macro")

print(f"\n=== Second corpus ({_src2}), n={len(_df2)} ===")
print(f"  bullying prevalence (gold): {_gold.mean():.3f}")
print(f"  accuracy   : {_acc:.3f}")
print(f"  precision  : {_prec:.3f}  (bullying class)")
print(f"  recall     : {_rec:.3f}  (bullying class)")
print(f"  F1         : {_f1:.3f}  (bullying class)")
print(f"  macro-F1   : {_macro_f1:.3f}")

# ---- bootstrap CIs (2000x, paired over the SAME rows so our model and the shipped
# LLM columns are resampled together and the deltas are comparable) ----
def _boot2(gold, pred, n_boot=2000, seed=0):
    _rng = np.random.default_rng(seed); _m = len(gold); _ix = np.arange(_m)
    _a = np.empty(n_boot); _p = np.empty(n_boot); _r = np.empty(n_boot); _f = np.empty(n_boot)
    for _k in range(n_boot):
        _s = _rng.choice(_ix, _m, replace=True)
        _g, _q = gold[_s], pred[_s]
        _a[_k] = (_g == _q).mean()
        _tp = float(((_g == 1) & (_q == 1)).sum())
        _p[_k] = _tp / max(float((_q == 1).sum()), 1.0)
        _r[_k] = _tp / max(float((_g == 1).sum()), 1.0)
        _f[_k] = 0.0 if (_p[_k] + _r[_k]) == 0 else 2 * _p[_k] * _r[_k] / (_p[_k] + _r[_k])
    _q25 = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return dict(acc=_q25(_a), prec=_q25(_p), rec=_q25(_r), f1=_q25(_f))

_ci2 = _boot2(_gold, _pred)
print(f"  95% CI (2000x item bootstrap):")
print(f"    accuracy  [{_ci2['acc'][0]:.3f}, {_ci2['acc'][1]:.3f}]")
print(f"    precision [{_ci2['prec'][0]:.3f}, {_ci2['prec'][1]:.3f}]")
print(f"    recall    [{_ci2['rec'][0]:.3f}, {_ci2['rec'][1]:.3f}]   <- primary deployment endpoint")
print(f"    F1        [{_ci2['f1'][0]:.3f}, {_ci2['f1'][1]:.3f}]")
# normal-class false-positive rate: the other half of the deployment pair
_fpr = float(((_gold == 0) & (_pred == 1)).sum() / max((_gold == 0).sum(), 1))
_fpr_ci = _boot2(1 - _gold, _pred)["rec"]     # recall of pred==1 among gold==0
print(f"  normal-content FPR: {_fpr:.3f}  95% CI [{_fpr_ci[0]:.3f}, {_fpr_ci[1]:.3f}]")
if "record_metrics" in dir():
    record_metrics(second_corpus_recall_ci_lo=_ci2["rec"][0], second_corpus_recall_ci_hi=_ci2["rec"][1],
                   second_corpus_prec_ci_lo=_ci2["prec"][0], second_corpus_prec_ci_hi=_ci2["prec"][1],
                   second_corpus_f1_ci_lo=_ci2["f1"][0], second_corpus_f1_ci_hi=_ci2["f1"][1],
                   second_corpus_normal_fpr=_fpr,
                   second_corpus_normal_fpr_ci_lo=_fpr_ci[0], second_corpus_normal_fpr_ci_hi=_fpr_ci[1])
# The corpus ships gpt_pred / gemini_pred / claude_pred columns; they are scored
# on these exact rows a few lines below, so the comparison is like-for-like.

_df2[[_text_col, "_gold_bin", "pred_idx", "pred_bin"]].to_csv(
    "/kaggle/working/second_corpus_results.csv", index=False)
# like-for-like: if the file ships LLM predictions, score them on the SAME rows
_llm_cols = [c for c in SECOND_CORPUS_LLM_COLS if c in _df2.columns]
if _llm_cols:
    print("\n=== LLM predictions shipped with the corpus, scored on the same rows ===")
    print(f"  {'model':22s} {'acc':>6} {'prec':>6} {'rec':>6} {'F1':>6}")
    print(f"  {'our fine-tuned BERT':22s} {_acc:6.3f} {_prec:6.3f} {_rec:6.3f} {_f1:6.3f}"
          f"   (n={len(_gold)})  recall CI [{_ci2['rec'][0]:.3f}, {_ci2['rec'][1]:.3f}]")
    print("  NOTE: gpt/gemini/claude predictions ship WITH the corpus (Muminovic 2025);")
    print("        prompts, model versions and decoding settings are his, not ours - cite the source paper.")
    for _lc in _llm_cols:
        _lp = _df2[_lc].map(_to_binary_bully)
        _mask = _lp.notna().to_numpy()
        _lp = _lp[_mask].astype(int).to_numpy(); _lg = _gold[_mask]
        _la = float((_lg == _lp).mean())
        _lpr, _lre, _lf1, _ = precision_recall_fscore_support(_lg, _lp, average="binary", zero_division=0)
        _lci = _boot2(_lg, _lp)
        print(f"  {_lc:22s} {_la:6.3f} {_lpr:6.3f} {_lre:6.3f} {_lf1:6.3f}   (n={_mask.sum()})"
              f"  recall CI [{_lci['rec'][0]:.3f}, {_lci['rec'][1]:.3f}]")
        if "record_metrics" in dir():
            record_metrics(**{f"second_corpus_{_lc}_f1": float(_lf1),
                              f"second_corpus_{_lc}_acc": float(_la)})
else:
    print("\nNo LLM prediction columns found in this file - compare against the published "
          "GPT-4.1 / Gemini / Claude numbers from the source paper.")

if "record_metrics" in dir():
    record_metrics(second_corpus_source=_src2, second_corpus_n=len(_df2),
                   second_corpus_acc=_acc, second_corpus_precision=_prec,
                   second_corpus_recall=_rec, second_corpus_f1=_f1,
                   second_corpus_macro_f1=_macro_f1)
print("\nSaved -> second_corpus_results.csv")


In [ ]:
# =============================================================
# GEMMA 2B ROUTING CELL
# Add gemma2-2b-it from Kaggle Models before running this.
# This replaces the Ollama llm_classify() function in app.py.
# Run this cell BEFORE the %%writefile app.py cell.
# =============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
# Strip markdown fences if present
import re, json

GEMMA_PATH = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/1"

print("Loading Gemma 2B-IT...")
gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH)
gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
gemma_model.eval()
print("Gemma loaded.")

def gemma_classify(text: str) -> dict | None:
    """
    Classifies a sentence using Gemma 2B-IT.
    Returns same schema as bert_classify for consistent downstream handling.
    Called only when BERT is uncertain or surface flags fire.
    """
    prompt = f"""<start_of_turn>user
You are a content moderation assistant. Classify the following sentence into exactly one of:
- hate speech: targets a person or group to degrade or dehumanize based on identity
- offensive: demeaning or hostile toward a specific person, but not identity-based hate  
- normal: banter, situational sarcasm, self-deprecating humor, meta-commentary, or neutral

Rules:
- Misspelled slurs or insults (loooser, stuuupid) count the same as correctly spelled ones
- Sarcasm at a situation (Monday, traffic, printer) = normal
- Sarcasm directed at a specific person = offensive
- ALL CAPS aggressive phrasing toward a person = offensive
- If harsh but targeting no one = normal

Sentence: "{text}"

Respond in this exact JSON format and nothing else:
{{"verdict": "hate speech" or "offensive" or "normal", "confidence": 0.0-1.0, "reason": "one sentence"}}<end_of_turn>
<start_of_turn>model
"""

    try:
        inputs = gemma_tokenizer(prompt, return_tensors="pt").to(gemma_model.device)
        with torch.no_grad():
            output = gemma_model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=gemma_tokenizer.eos_token_id,
            )
        generated = output[0][inputs["input_ids"].shape[1]:]
        raw = gemma_tokenizer.decode(generated, skip_special_tokens=True).strip()

        
        raw = re.sub(r"```json|```", "", raw).strip()
        
        # Find the JSON object
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if not match:
            return None
            
        result = json.loads(match.group())
        verdict_map = {"hate speech": "hate", "offensive": "offensive", "normal": "normal"}
        verdict = verdict_map.get(result.get("verdict", "").lower(), "normal")

        return {
            "verdict":        verdict,
            "confidence":     float(result.get("confidence", 0.80)),
            "scores":         {verdict: float(result.get("confidence", 0.80))},
            "tone":           result.get("reason", "LLM classification"),
            "desc_type":      "LLM-routed classification",
            "desc_text":      result.get("reason", ""),
            "desc_certainty": "detected",
            "reason":         f"Routed to Gemma reasoning layer. {result.get('reason', '')}",
            "word_importances": []
        }
    except Exception as e:
        print(f"Gemma classify error: {e}")
        return None


# Quick sanity check
test_sentences = [
    "Hey! I think you are fucking loooser",
    "You are such a MORON honestly",
    "Oh great, another Monday. Just what I always wanted.",
]

print("\nGemma routing sanity check:")
print(f"{'TEXT':<50} {'VERDICT':<12} {'CONF':>6}  REASON")
print("─" * 90)
for s in test_sentences:
    result = gemma_classify(s)
    if result:
        print(f"{s[:48]:<50} {result['verdict'].upper():<12} {result['confidence']*100:>5.1f}%  {result['tone'][:40]}")
    else:
        print(f"{s[:48]:<50} FAILED")

In [ ]:
# ============================================================
# DOES ROUTING HELP?  -  BERT vs Gemma on the deferred slice
# ============================================================
# The deferral threshold is only worth having if Gemma actually beats BERT on
# the cases BERT is unsure about. Take the held-out TEST rows whose calibrated
# BERT confidence < route_threshold, run gemma_classify on a bounded sample of
# them, and compare both to gold. Writes the *measured* Gemma accuracy back into
# calibration.json (replacing the GEMMA_ASSUMED_ACCURACY guess).
# ------------------------------------------------------------
MAX_GEMMA = 600          # cap the Gemma pass (each call is a generate()); tune for time budget

import os, json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

if "gemma_classify" not in dir():
    raise SystemExit("Run the GEMMA 2B ROUTING cell first.")

_MDLDIR = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LAB2ID = {"hate": 0, "offensive": 1, "normal": 2}
ID2LAB = {v: k for k, v in LAB2ID.items()}

_cal = {"temperature": 1.0, "route_threshold": 0.60}
_cpath = os.path.join(_MDLDIR, "calibration.json")
try:
    _cal.update(json.load(open(_cpath)))
except FileNotFoundError:
    print("no calibration.json - using T=1.0, threshold=0.60")
T_SCALE = float(_cal["temperature"]); ROUTE = float(_cal["route_threshold"])
print(f"temperature={T_SCALE:.3f} | route_threshold={ROUTE:.3f}")

test_df = load_eval_split("test").reset_index(drop=True)
_tok = AutoTokenizer.from_pretrained(_MDLDIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()


@torch.no_grad()
def _bert(texts, bs=64):
    conf, pred = [], []
    for i in range(0, len(texts), bs):
        enc = _tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                   padding=True, max_length=128).to(_DEV)
        p = (_mdl(**enc).logits / T_SCALE).softmax(-1).cpu().numpy()
        conf.extend(p.max(1)); pred.extend(p.argmax(1))
    return np.asarray(conf), np.asarray(pred)


conf, bert_pred = _bert(test_df["text"].tolist())
gold = test_df["clean_label"].to_numpy()
deferred = conf < ROUTE
bert_only_acc = float((bert_pred == gold).mean())
print(f"\nTEST: {len(test_df)} rows | BERT-only acc {bert_only_acc:.3f} | "
      f"deferred (conf < {ROUTE:.2f}): {int(deferred.sum())} ({deferred.mean() * 100:.1f}%)")

if deferred.sum() == 0:
    raise SystemExit("No rows fall below route_threshold on the test set - nothing to route.")

sub = test_df[deferred].copy()
sub["_bp"] = bert_pred[deferred]
sub["_gold"] = gold[deferred]
if len(sub) > MAX_GEMMA:
    sub = (sub.groupby("_gold", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), max(1, round(MAX_GEMMA * len(g) / deferred.sum()))),
                                        random_state=42)))
    print(f"  sampling {len(sub)} of {int(deferred.sum())} deferred rows for the Gemma pass")

gem = []
for _i, t in enumerate(sub["text"].tolist()):
    r = gemma_classify(t)
    gem.append(LAB2ID.get(r["verdict"], -1) if r else -1)
    if (_i + 1) % 100 == 0:
        print(f"  gemma {_i + 1}/{len(sub)}", flush=True)
gem = np.asarray(gem)
ok = gem >= 0
g = sub["_gold"].to_numpy(); b = sub["_bp"].to_numpy()

bert_acc_def = float((b[ok] == g[ok]).mean())
gem_acc_def = float((gem[ok] == g[ok]).mean())
proj_cascade = bert_only_acc + deferred.mean() * (gem_acc_def - bert_acc_def)

print(f"\n--- deferred slice (n={int(ok.sum())} scored, {int((~ok).sum())} Gemma parse-fails) ---")
print(f"  BERT  accuracy : {bert_acc_def:.3f}")
print(f"  Gemma accuracy : {gem_acc_def:.3f}")
print(f"  delta (Gemma - BERT): {gem_acc_def - bert_acc_def:+.3f}")
print(f"\n--- whole test set ---")
print(f"  BERT-only               : {bert_only_acc:.3f}")
print(f"  projected BERT->Gemma   : {proj_cascade:.3f}   (delta {proj_cascade - bert_only_acc:+.3f})")
_v = ("Routing HELPS" if gem_acc_def > bert_acc_def + 0.01 else
      "Routing does NOT help" if gem_acc_def < bert_acc_def - 0.01 else "Routing is a WASH")
print(f"\n>>> {_v} on this test set (measured, not assumed). <<<")

# split by gold class: does the LLM help on hateful-class recall specifically, and not
# on the non-hate contrasts?
print("\n--- deferred slice, split by gold class ---")
_by_class = {}
for _cid, _cname in ID2LAB.items():
    _m = (g == _cid) & ok
    if _m.sum() == 0:
        continue
    _ba = float((b[_m] == g[_m]).mean())
    _ga = float((gem[_m] == g[_m]).mean())
    _by_class[_cname] = {"n": int(_m.sum()), "bert_acc": _ba, "gemma_acc": _ga, "delta": _ga - _ba}
    print(f"  {_cname:10s} (n={int(_m.sum()):4d}): BERT {_ba:.3f} | Gemma {_ga:.3f} | delta {_ga - _ba:+.3f}")

# ASYMMETRIC deferral: take Gemma's verdict only when it says "hate" (id 0), keep BERT
# otherwise - Gemma escalates almost everything, so this keeps its hate-recall win while
# dropping its offensive/normal damage.
casc_asym = b.copy()
_use_gem = ok & (gem == 0)
casc_asym[_use_gem] = gem[_use_gem]
asym_acc_def = float((casc_asym == g).mean())
proj_asym = bert_only_acc + deferred.mean() * (asym_acc_def - float((b == g).mean()))
print(f"\n--- ASYMMETRIC (Gemma only on 'hate', else BERT) ---")
print(f"  deferred-slice acc : BERT {float((b == g).mean()):.3f} -> asym {asym_acc_def:.3f} "
      f"(delta {asym_acc_def - float((b == g).mean()):+.3f}; used Gemma on {int(_use_gem.sum())}/{len(g)})")
print(f"  projected whole-test: BERT-only {bert_only_acc:.3f} -> asym cascade {proj_asym:.3f} "
      f"(delta {proj_asym - bert_only_acc:+.3f})")

_cal["gemma_measured_acc_deferred"] = gem_acc_def
_cal["bert_acc_deferred"] = bert_acc_def
_cal["routing_delta"] = gem_acc_def - bert_acc_def
json.dump(_cal, open(_cpath, "w"), indent=2)
out = pd.DataFrame({"text": sub["text"].to_numpy(), "gold": [ID2LAB[i] for i in g],
                    "bert": [ID2LAB[i] for i in b],
                    "gemma": [ID2LAB[i] if i >= 0 else "PARSE_FAIL" for i in gem]})
out.to_csv("/kaggle/working/routing_experiment_cases.csv", index=False)
if "record_metrics" in dir():
    record_metrics(routing_bert_acc_deferred=bert_acc_def, routing_gemma_acc_deferred=gem_acc_def,
                   routing_delta=gem_acc_def - bert_acc_def,
                   routing_deferred_frac=float(deferred.mean()), proj_cascade_acc=proj_cascade,
                   routing_asym_acc_deferred=asym_acc_def, proj_asym_cascade_acc=proj_asym,
                   routing_gemma_parse_fail=int((~ok).sum()),
                   routing_gemma_parse_fail_rate=float((~ok).mean()))
    for _cname, _v2 in _by_class.items():
        record_metrics(**{f"routing_delta_{_cname}": _v2["delta"],
                          f"routing_n_{_cname}": _v2["n"]})
print("\nSaved -> routing_experiment_cases.csv ; measured accuracy written to calibration.json")


In [ ]:
# ============================================================
# BASELINES  -  what the fine-tune is compared against
# ============================================================
#   B0  off-the-shelf tum-nlp/bert-hateXplain (no fine-tuning)
#   B1  Gemma-2B-it zero-shot (gemma_classify), sampled
# Both on the held-out TEST split and on HateCheck (primary "hate_only" mapping).
# OUR fine-tune numbers are read back from run_metrics.json.
# ------------------------------------------------------------
MAX_GEMMA_TEST = 400          # Gemma = one generate() per row; raise for tighter numbers
MAX_GEMMA_HC   = 400

import os, json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score

if "gemma_classify" not in dir():
    raise SystemExit("Run the GEMMA 2B ROUTING cell first.")

_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUR2ID = {"hate": 0, "offensive": 1, "normal": 2}
ID2OUR = {v: k for k, v in OUR2ID.items()}
test_df = load_eval_split("test").reset_index(drop=True)
_gold = test_df["clean_label"].to_numpy()


@torch.no_grad()
def _argmax(model, tok, texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), return_tensors="pt", truncation=True,
                  padding=True, max_length=128).to(_DEV)
        out.extend(model(**enc).logits.argmax(-1).cpu().tolist())
    return out


# ---- B0: off-the-shelf bert-hateXplain ----------------------------------
_bt = AutoTokenizer.from_pretrained("tum-nlp/bert-hateXplain")
_bm = AutoModelForSequenceClassification.from_pretrained("tum-nlp/bert-hateXplain").to(_DEV).eval()
_id2lab = {int(k): str(v).lower() for k, v in _bm.config.id2label.items()}

def _to_ours(name):
    n = name.lower()
    if "hate" in n: return 0
    if "offens" in n: return 1
    if any(w in n for w in ("normal", "neither", "none", "non-toxic")): return 2
    return None
_perm = {bi: _to_ours(nm) for bi, nm in _id2lab.items()}
if None in _perm.values() or set(_perm.values()) != {0, 1, 2}:
    if len(_id2lab) != 2:   # a binary head is handled by the _B0_BINARY branch, not "unmappable"
        print("  base id2label unmappable:", _id2lab, "-> assuming HateXplain order [hatespeech, normal, offensive]")
    _perm = {0: 0, 1: 2, 2: 1}

_B0_BINARY = len(_id2lab) == 2   # tum-nlp/bert-hateXplain is {0:non-toxic, 1:toxic} - a binary head
if _B0_BINARY:
    # NOTE: "non-toxic" also contains "tox" - without the non/not guard this
    # resolves to 0 and every B0 prediction is inverted (that was acc 0.232).
    _tox_idx = next((i for i, nm in _id2lab.items()
                     if ("tox" in nm or "hate" in nm or "abus" in nm)
                     and not nm.startswith(("non", "not"))), 1)
    print(f"  B0 toxic index -> {_tox_idx} ({_id2lab[_tox_idx]})")
    _b0_raw = _argmax(_bm, _bt, test_df["text"].tolist())
    _b0_bin = np.array([1 if i == _tox_idx else 0 for i in _b0_raw])
    _gold_bin = np.array([0 if c == 2 else 1 for c in _gold])   # {hate,offensive}->1, normal->0
    b0_test_f1 = f1_score(_gold_bin, _b0_bin, average="macro")
    b0_test_bin_acc = float((_gold_bin == _b0_bin).mean())
    print(f"  B0 is a BINARY head {_id2lab} - scored binary: acc {b0_test_bin_acc:.3f}, macroF1 {b0_test_f1:.3f}")
else:
    b0_test_f1 = f1_score(_gold, [_perm[i] for i in _argmax(_bm, _bt, test_df["text"].tolist())], average="macro")
    b0_test_bin_acc = float("nan")
b0_hc = float("nan")
try:
    _hc = pd.read_csv("/kaggle/working/hatecheck_results.csv")
    _hr = _argmax(_bm, _bt, _hc["text"].tolist())
    if _B0_BINARY:
        _pb = np.array(["hateful" if i == _tox_idx else "non-hateful" for i in _hr])
        b0_hc = float((_pb == _hc["label_gold"].to_numpy()).mean())
        _hp = None
    else:
        _hp = [_perm[i] for i in _hr]
    if _hp is not None:
        _pb = np.array(["hateful" if ID2OUR[i] == "hate" else "non-hateful" for i in _hp])
        b0_hc = float((_pb == _hc["label_gold"].to_numpy()).mean())
except Exception as e:
    print("  B0 HateCheck skipped:", repr(e))
del _bm

# ---- B1: Gemma-2B zero-shot (sampled) ----------------------------------
def _gem_eval(frame, textcol, goldcol, cap, kind):
    d = frame.groupby(goldcol, group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, round(cap * len(g) / len(frame)))), random_state=42))
    pr = []
    for j, t in enumerate(d[textcol].tolist()):
        r = gemma_classify(t)
        pr.append(OUR2ID.get(r["verdict"], -1) if r else -1)
        if (j + 1) % 100 == 0: print(f"  gemma-{kind} {j + 1}/{len(d)}", flush=True)
    pr = np.array(pr); ok = pr >= 0
    return d, pr, ok

_dt, _gp, _ok = _gem_eval(test_df.assign(_g=_gold), "text", "_g", MAX_GEMMA_TEST, "test")
b1_test_f1 = f1_score(_dt["_g"].to_numpy()[_ok], _gp[_ok], average="macro")
b1_fail = int((~_ok).sum())
b1_hc = float("nan")
try:
    _hc = pd.read_csv("/kaggle/working/hatecheck_results.csv")
    _hd, _hp, _hok = _gem_eval(_hc, "text", "label_gold", MAX_GEMMA_HC, "hc")
    _pb = np.where(_hp[_hok] == OUR2ID["hate"], "hateful", "non-hateful")
    b1_hc = float((_pb == _hd["label_gold"].to_numpy()[_hok]).mean())
except Exception as e:
    print("  B1 HateCheck skipped:", repr(e))

# ---- comparison table -------------------------------------------------
_rm = json.load(open("/kaggle/working/run_metrics.json")) if os.path.exists("/kaggle/working/run_metrics.json") else {}
tab = pd.DataFrame([
    ["off-the-shelf bert-hateXplain",       b0_test_f1, b0_hc],
    [f"Gemma-2B zero-shot (~{MAX_GEMMA_TEST}/{MAX_GEMMA_HC})", b1_test_f1, b1_hc],
    ["our fine-tune",                        _rm.get("test_macro_f1", float("nan")),
                                             _rm.get("hatecheck_overall_hateonly", float("nan"))],
], columns=["model", "test_macroF1", "hatecheck_overall(hate_only)"])
print("\n=== BASELINE COMPARISON ===")
print(tab.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"(Gemma test parse-failures: {b1_fail}/{len(_dt)})")
tab.to_csv("/kaggle/working/baselines.csv", index=False)
if "record_metrics" in dir():
    record_metrics(baseline_offshelf_test_f1=float(b0_test_f1), baseline_offshelf_hc=float(b0_hc),
                   baseline_offshelf_binary=int(_B0_BINARY),
                   baseline_gemma_test_f1=float(b1_test_f1), baseline_gemma_hc=float(b1_hc),
                   baseline_gemma_parse_fail=int(b1_fail))
print("Saved -> baselines.csv")


In [ ]:
# ============================================================
# LATENCY  -  ms/query: BERT-only path vs the BERT->Gemma cascade
# ============================================================
# The premise of a cascade is efficiency. Per-query wall time for BERT inference,
# the normalize_text() pre-pass, and one Gemma call, combined with the measured
# routed fraction into an expected cascade latency.
# ------------------------------------------------------------
N_LAT = 200

import time, json, sys, importlib, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_MDLDIR = LOCAL_MODEL_DIR if "LOCAL_MODEL_DIR" in dir() else "best_model"
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_sample = load_eval_split("test")["text"].astype(str).tolist()[:N_LAT]

_tok = AutoTokenizer.from_pretrained(_MDLDIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(_MDLDIR).to(_DEV).eval()


@torch.no_grad()
def _bert_once(t):
    enc = _tok(t, return_tensors="pt", truncation=True, padding=True, max_length=128).to(_DEV)
    _mdl(**enc).logits.softmax(-1)
    if _DEV.type == "cuda":
        torch.cuda.synchronize()


for t in _sample[:10]:
    _bert_once(t)                       # warmup
_bt = []
for t in _sample:
    _s = time.perf_counter(); _bert_once(t); _bt.append((time.perf_counter() - _s) * 1000)
bert_ms, bert_p95 = float(np.median(_bt)), float(np.percentile(_bt, 95))

sys.path.insert(0, "/kaggle/working")
import textnorm; importlib.reload(textnorm)
_nt = []
for t in _sample:
    _s = time.perf_counter(); textnorm.normalize_text(t); _nt.append((time.perf_counter() - _s) * 1000)
norm_ms = float(np.median(_nt))

gem_ms = float("nan")
if "gemma_classify" in dir():
    _gt = []
    for t in _sample[:20]:
        _s = time.perf_counter(); gemma_classify(t); _gt.append((time.perf_counter() - _s) * 1000)
    gem_ms = float(np.median(_gt))

routed = 0.0
for _src in (f"{_MDLDIR}/calibration.json", "/kaggle/working/run_metrics.json"):
    try:
        _j = json.load(open(_src))
        routed = float(_j.get("test_routed_frac_at_threshold",
                              _j.get("routing_deferred_frac", routed)))
        if routed:
            break
    except Exception:
        pass

_bert_path = bert_ms + norm_ms
cascade_ms = _bert_path + routed * (gem_ms if gem_ms == gem_ms else 0.0)
print(f"\nBERT inference      : {bert_ms:7.1f} ms/query   (p95 {bert_p95:.1f})")
print(f"normalize_text()    : {norm_ms:7.2f} ms/query")
print("Gemma (one call)    : " + (f"{gem_ms:7.0f} ms/query" if gem_ms == gem_ms else "    n/a"))
print(f"routed fraction     : {routed:.3f}")
print(f"BERT-only path      : {_bert_path:7.1f} ms/query")
print(f"BERT->Gemma cascade : {cascade_ms:7.1f} ms/query   ({cascade_ms / max(_bert_path, 1e-6):.1f}x BERT-only)")
if "record_metrics" in dir():
    record_metrics(latency_bert_ms=bert_ms, latency_bert_p95_ms=bert_p95, latency_norm_ms=norm_ms,
                   latency_gemma_ms=gem_ms, latency_cascade_ms=cascade_ms, latency_routed_frac=routed)
del _mdl


In [ ]:
# ============================================================
# ABLATION LEDGER  -  one row per run; flip flags in CACHE CONFIG, re-run, compare
# ============================================================
# Suggested sweep (change RUN_LABEL each time):
#   soft_off        : SOFT_LABEL_TRAINING = False
#   soft_uw1        : SOFT_DISAGREEMENT_UPWEIGHT = 1.0
#   soft_uw6        : SOFT_DISAGREEMENT_UPWEIGHT = 6.0   (current)
#   no_contrastive  : remove the contrastive CSV path in the dataset cell
# ------------------------------------------------------------
import os, json, datetime, pandas as pd

_LEDGER = "/kaggle/working/ablation_results.csv"
_REPO = MODEL_REPO if "MODEL_REPO" in dir() else None
try:
    from huggingface_hub import hf_hub_download
    prior = pd.read_csv(hf_hub_download(_REPO, "ablation_results.csv", repo_type="model"))
    print(f"pulled {len(prior)} prior ledger row(s) from {_REPO}")
except Exception as e:
    prior = pd.read_csv(_LEDGER) if os.path.exists(_LEDGER) else pd.DataFrame()
    print("no prior ledger on the Hub:", repr(e)[:100])

row = json.load(open("/kaggle/working/run_metrics.json")) if os.path.exists("/kaggle/working/run_metrics.json") else {}
row.setdefault("run_label", RUN_LABEL if "RUN_LABEL" in dir() else "run")
row["run_at"] = datetime.datetime.utcnow().isoformat(timespec="seconds")

ledger = pd.concat([prior, pd.DataFrame([row])], ignore_index=True)
ledger = ledger.drop_duplicates(subset=["run_id"], keep="last").reset_index(drop=True)
_front = ["run_label", "soft_label_training", "soft_disagreement_upweight", "contrastive_rows",
          "dev_macro_f1", "test_macro_f1", "test_spectrum_entcorr", "route_threshold",
          "routing_delta", "hatecheck_overall_hateonly", "hatecheck_overall_norm",
          "norm_recovery_mean", "baseline_offshelf_test_f1", "baseline_gemma_test_f1", "run_at"]
ledger = ledger[[c for c in _front if c in ledger.columns]
                + [c for c in ledger.columns if c not in _front]]
ledger.to_csv(_LEDGER, index=False)
print("\n=== ABLATION LEDGER ===")
with pd.option_context("display.max_columns", None, "display.width", 220):
    print(ledger.to_string(index=False))
try:
    from huggingface_hub import upload_file
    upload_file(path_or_fileobj=_LEDGER, path_in_repo="ablation_results.csv", repo_id=_REPO)
    print("\npushed ledger ->", _REPO)
except Exception as e:
    print("\nledger push skipped:", repr(e))


In [ ]:
# ============================================================
# SOFT-LABEL ANALYSIS  -  does disagreement modelling do anything?
# ============================================================
# Reads the ablation ledger (needs >= 2 runs, at least one soft_off). Compares
# soft vs hard on hard metrics (macro-F1) AND disagreement calibration
# (test entropy correlation / KL). Built to surface a negative result if that is
# what the numbers say. Skips quietly (no halt) until the ledger has enough rows.
# ------------------------------------------------------------
import os, numpy as np, pandas as pd


def _soft_label_analysis():
    _LEDGER = "/kaggle/working/ablation_results.csv"
    try:
        from huggingface_hub import hf_hub_download
        L = pd.read_csv(hf_hub_download(MODEL_REPO, "ablation_results.csv", repo_type="model"))
    except Exception:
        L = pd.read_csv(_LEDGER) if os.path.exists(_LEDGER) else pd.DataFrame()

    _have = [str(r) for r in L.get("run_label", [])]
    _need = [r for r in ("main", "soft_off", "no_contrastive", "soft_uw1")
             if not any(str(h).startswith(r) for h in _have)]
    if len(L) < 2 or "soft_label_training" not in L.columns or "test_macro_f1" not in L.columns:
        print("=" * 62)
        print("ABLATION LEDGER INCOMPLETE - this is expected until all 3 runs finish.")
        print(f"  have : {_have or '(none)'}")
        print(f"  still needed: {_need}")
        print("  To add one: set RUN_LABEL at the top of the CACHE CONFIG cell,")
        print("  then Save & Run All (Commit). The ledger is pushed to the HF repo")
        print("  after every run, so rows accumulate across separate commits.")
        print("=" * 62)
        return

    _cols = [c for c in ["run_label", "soft_label_training", "soft_disagreement_upweight",
                         "dev_macro_f1", "test_macro_f1", "test_spectrum_entcorr",
                         "test_spectrum_kl", "hatecheck_overall_hateonly"] if c in L.columns]
    tab = L[_cols].sort_values(["soft_label_training", "soft_disagreement_upweight"], na_position="first")
    print("=== runs ===")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    hard = L[L["soft_label_training"] == 0]
    soft = L[L["soft_label_training"] == 1]
    _has_ec = "test_spectrum_entcorr" in L.columns
    if len(hard) and len(soft):
        h = hard.iloc[-1]
        sb = soft.loc[soft["test_macro_f1"].idxmax()]
        _ec = lambda r: f" | entropy_corr {r.get('test_spectrum_entcorr', float('nan')):.3f}" if _has_ec else ""
        print(f"\nhard-label ({h.get('run_label', '?')}): test F1 {h['test_macro_f1']:.4f}{_ec(h)}")
        print(f"best soft   ({sb.get('run_label', '?')}): test F1 {sb['test_macro_f1']:.4f}{_ec(sb)}")
        d_f1 = sb["test_macro_f1"] - h["test_macro_f1"]
        line = f"\nsoft - hard:  dF1 {d_f1:+.4f}"
        if _has_ec:
            d_ec = float(sb.get("test_spectrum_entcorr", np.nan)) - float(h.get("test_spectrum_entcorr", np.nan))
            print(line + f"  |  d(entropy_corr) {d_ec:+.3f}")
            v = ("soft-label modelling MEANINGFULLY improves disagreement calibration"
                 if d_ec > 0.10 else
                 "soft-label modelling does NOT improve disagreement calibration (entropy_corr ~flat) -> "
                 "report as a negative result or drop the spectrum framing")
            print(f">>> {v} <<<")
        else:
            print(line)

    if _has_ec and soft["soft_disagreement_upweight"].nunique() >= 3:
        print("\nentropy_corr vs SOFT_DISAGREEMENT_UPWEIGHT:")
        for _, r in soft.sort_values("soft_disagreement_upweight").iterrows():
            print(f"  uw={r['soft_disagreement_upweight']:5.1f}  "
                  f"entropy_corr={r['test_spectrum_entcorr']:.3f}  test_F1={r['test_macro_f1']:.4f}")


_soft_label_analysis()


In [ ]:
%%writefile app.py

from flask import Flask, request, jsonify, render_template
import torch
import torch.nn.functional as F
import re, requests, json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, AutoConfig
from spellchecker import SpellChecker

app = Flask(__name__)

MODEL_PATH = "best_model"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(DEVICE)
model.eval()

LABELS    = {0: "hate", 1: "offensive", 2: "normal"}
LABEL_IDX = {"hate": 0, "offensive": 1, "normal": 2}

# -- Calibration: temperature + deferral threshold (best_model/calibration.json) --
# Written by the CONFIDENCE CALIBRATION cell; falls back to identity if absent.
_CALIB = {"temperature": 1.0, "route_threshold": 0.60}
try:
    with open(f"{MODEL_PATH}/calibration.json") as _cf:
        _CALIB.update(json.load(_cf))
    print(f"Calibration loaded: T={_CALIB['temperature']}, "
          f"route to Gemma when confidence < {_CALIB['route_threshold']}")
except FileNotFoundError:
    print("No calibration.json - using T=1.0 and route threshold 0.60")
TEMPERATURE     = float(_CALIB["temperature"])
ROUTE_THRESHOLD = float(_CALIB["route_threshold"])

# ── Tone ─────────────────────────────────────────────────────
def get_tone(h, o, n):
    if n >= 0.85:
        return "clearly fine"
    elif n >= 0.65:
        return "normal with a hint of edge" if (h + o) > 0.2 else "normal"
    elif n >= 0.45 and o >= 0.35:
        return "ambiguous — banter or mild offense, context-dependent"
    elif o >= 0.60:
        return "likely offensive"
    elif h >= 0.50:
        return "likely hate speech"
    elif o >= 0.40 and h >= 0.25:
        return "offensive with hateful undertones"
    else:
        return "uncertain — review manually"


# ── Self-directed distress layer (Direction 2 — keyword pre-filter, no retraining) ──
# Regex word boundaries (not raw substrings) plus a required first-person pronoun
# elsewhere in the message: "nobody cares about the deadline" and "how do I matter
# to this project" no longer fire the way a bare substring check did. Trade-off: a
# terse "nobody cares." with no other first-person word in the message no longer
# triggers either - narrower recall for fewer false positives, intentional given
# this only ever surfaces an additive support resource, never a verdict.
_FIRST_PERSON_RE = re.compile(r"\b(i|i'm|im|i've|ive|me|my|myself)\b")
_DISTRESS_RE = [re.compile(p) for p in [
    r"\bwish to give up\b",
    r"\bwant to give up\b",
    r"\bdon'?t think i matter\b",
    r"\bdo i (?:truly )?matter\b(?!\s+to\b)",
    r"\bfeel like giving up\b",
    r"\b(?:nobody|no one) cares\b(?!\s+about\s+(?:the|this|that|a|an)\b)",
    r"\bwhat'?s (?:the |even the )?point\b",
    r"\bi don'?t belong\b",
    r"\btired of everything\b",
    r"\btired of living\b",
    r"\bcan'?t go on\b",
    r"\bno one would notice\b",
    r"\bbetter off without me\b",
]]

def check_self_distress(text):
    t = text.lower()
    if not _FIRST_PERSON_RE.search(t):
        return False
    return any(p.search(t) for p in _DISTRESS_RE)

# -- Support resource (ADDITIVE, non-overriding) -- NOT a diagnosis or assessment --
# Some phrasings are commonly associated with self-directed distress. When one is
# present we ATTACH a support-resource block to the normal response; the verdict is
# unchanged and no claim is made about the person's state. No confidence reported.
# Framing follows CLPsych shared-task ethics guidance.
SUPPORT_RESOURCES = [
    {"name": "988 Suicide & Crisis Lifeline (US)", "contact": "call or text 988",
     "url": "https://988lifeline.org"},
    {"name": "Crisis Text Line (US / CA / UK / IE)", "contact": "text HOME to 741741",
     "url": "https://www.crisistextline.org"},
    {"name": "Befrienders Worldwide", "contact": "helpline directory by country",
     "url": "https://www.befrienders.org"},
]


def maybe_show_support_resource(text):
    """Additive. Returns a support-resource block to merge into the normal response
    when the message uses self-directed-distress wording, else None. No verdict,
    no score, no confidence."""
    # HARD GATE: the support card never fires on second-person abuse. Both the
    # keyword path AND the Gemma path must sit behind the first-person check -
    # previously the LLM path bypassed it and fired on threats/insults
    # ("count your days", "you are specifically not invited").
    if not _FIRST_PERSON_RE.search(text.lower()):
        return None
    if not (check_self_distress(text) or check_self_distress_llm(text)):
        return None
    return {
        "title": "If you're going through something, you don't have to face it alone",
        "body": ("This message uses wording sometimes associated with self-directed "
                 "distress. This is not an assessment of how you are doing \u2014 but if "
                 "you would like to talk to someone, here are some options."),
        "resources": SUPPORT_RESOURCES,
        "disclaimer": ("Not a clinical or crisis assessment. If you or someone else "
                       "may be in immediate danger, contact local emergency services."),
    }


def _respond(payload, text):
    """Single response path: attach the support resource if applicable, then jsonify."""
    sr = maybe_show_support_resource(text)
    if sr:
        payload = {**payload, "support_resource": sr}
    return jsonify(payload)



# ── Descriptive type classifier — keyword pools ────────────────
SITUATION_WORDS   = ["monday","tuesday","wednesday","thursday","friday","weekend",
                     "meeting","alarm","traffic","wifi","update","printer","weather",
                     "train","bus","flight","elevator","queue","internet","server"]
SELF_WORDS        = ["myself", "i'm such", "my own fault", "typical me",
                     "classic me", "i swear i", "i can't even",
                     "my cooking","my dog","my cat","my brain","my life"]
META_WORDS        = ["studies show","research says","posts with","comments that",
                     "people who use","content that","language like","words like",
                     "if you want","tend to","more likely","algorithm"]
BANTER_PATTERNS   = ["energy-saving","buffering","lazy","procrastin","not bad",
                     "could be worse","just saying","technically","in theory"]
BACKHANDED        = ["genius","einstein","rocket scientist","truly","wow","amazing",
                     "incredible","so smart","next level","well done","great job",
                     "nice job","figured out","managed to","finally"]
NEG_SARCASM       = ["so well","worked out","sure,","oh sure","oh great","because that",
                     "because clearly","obviously","naturally","of course","right,"]
CONDESCENDING     = ["not everyone","some people","bless","at least you tried",
                     "for someone like","given your","considering where","for your level",
                     "i'm sure you","surprisingly","not bad for"]
CODED_EXCL        = ["built for","cut out for","belong here","that kind of work",
                     "some groups","certain people","those communities","people like that",
                     "not meant for","just aren't","just not"]
STEREOTYPE        = ["they're just like","what do you expect","that's how they",
                     "typical","always been","never change","those people"]
RHETORICAL_ATK    = ["do you even","can you not","how could you","did you really",
                     "seriously?","really?","are you kidding","you actually"]
HOSTILE_BANTER    = ["personality of","has the iq of","brain of a","makes me question",
                     "existence is","presence is","company is","company like"]
IRONIC_PRAISE     = ["such a brave","so courageous","what a hero","legend","goat",
                     "truly inspiring","role model","we're blessed"]
DOG_WHISTLE       = ["dog whistle","certain people","you know who","those kind",
                     "you know how they","real americans","real people"]
RHETORICAL_PHIL   = ["rhetorical","ironic","sarcastic"]

NEGATORS = ["not", "n't", "never", "no", "hardly", "barely"]

def is_negated(text, keyword, window=3):
    """Crude check: was a negator within `window` words before the keyword?"""
    t = text.lower()
    idx = t.find(keyword)
    if idx == -1:
        return False
    before_words = t[:idx].split()[-window:]
    return any(neg in before_words for neg in NEGATORS)


def _grounded_score(matches, important_words, text):
    """Score = raw match count, +2 bonus per match that overlaps a high-importance
    word (i.e. a word BERT's masking pass actually leaned on), -1 penalty per
    match that's locally negated."""
    score = 0
    for m in matches:
        score += 1
        if any(iw in m or m in iw for iw in important_words):
            score += 2
        if is_negated(text, m):
            score -= 1
    return score


def detect_type_scored(verdict, scores, text, word_importances):
    """Rule-based fallback used only when the Gemma type-detect call is skipped
    or fails. Scores every candidate category instead of cascading if/elif, so
    match order no longer arbitrarily decides ties, and grounds matches against
    BERT's own word-importance signal rather than trusting bare substring hits."""
    t = text.lower()
    h, o, n = scores["hate"], scores["offensive"], scores["normal"]
    important_words = {w["word"].lower().strip(".,!?;:\"'") for w in word_importances[:5]}

    DESCRIPTIONS = {
        "Meta-commentary": "This sentence talks *about* offensive or harmful language rather than using it. The model correctly recognises it as analysis, not attack.",
        "Situational sarcasm": "Sarcasm aimed at a situation, event, or thing — not at a person. The frustration is real but no one is being targeted or demeaned.",
        "Self-deprecating humor": "The speaker is the subject of their own joke. Directing humor at yourself signals playfulness rather than hostility toward others.",
        "Banter / playful ribbing": "Light, mutual teasing with no real sting. The tone is playful rather than demeaning — the kind of thing said between friends.",
        "Rhetorical / philosophical": "An abstract or reflective observation about life, people, or the world. No specific target, no hostile intent.",
        "Backhanded compliment": "Uses positive or praising language as a vehicle to deliver a put-down. The surface sounds like a compliment but the intent is to belittle.",
        "Negative sarcasm (person-targeted)": "Heavy irony directed at a person to express contempt or dismissal. The exaggerated phrasing makes the real meaning obvious.",
        "Ironic praise": "Praise so over-the-top it inverts into mockery. The speaker uses admiration language to signal the opposite.",
        "Condescending dismissal": "Implies the target is beneath a standard or expectation without an outright insult. The offense is in the assumption of inferiority.",
        "Rhetorical question as attack": "Phrased as a question but functions as an insult or challenge. No real answer is expected.",
        "Hostile banter": "Started in the register of banter but crossed into genuine sting. Specific enough to feel targeted rather than playful.",
        "Coded exclusionary language": "Uses neutral-sounding words to imply a group's inferiority or unsuitability. No slurs, no explicit hostility.",
        "Stereotype reinforcement": "Treats a negative generalisation about a group as obvious fact.",
        "Dog-whistle language": "Phrases that sound innocuous on the surface but carry a specific hateful meaning within certain communities.",
    }

    candidates = []  # (name, score, certainty)

    def consider(name, wordlist, certainty="detected"):
        matches = [w for w in wordlist if w in t]
        if not matches:
            return
        s = _grounded_score(matches, important_words, t)
        if s > 0:
            candidates.append((name, s, certainty))

    if verdict == "normal":
        consider("Meta-commentary", META_WORDS)
        consider("Situational sarcasm", SITUATION_WORDS)
        consider("Self-deprecating humor", SELF_WORDS)
        consider("Banter / playful ribbing", BANTER_PATTERNS)
        consider("Rhetorical / philosophical", RHETORICAL_PHIL)

        if candidates:
            candidates.sort(key=lambda x: x[1], reverse=True)
            name, _, certainty = candidates[0]
            return {"type": name, "description": DESCRIPTIONS[name], "certainty": certainty}

        if (h + o) > 0.30:
            return {
                "type": "Ambiguous normal",
                "description": "The sentence reads as normal overall, but carries enough edge that context could shift its meaning.",
                "certainty": "possible"
            }
        return {
            "type": "Neutral statement",
            "description": "A straightforward sentence with no detectable hostility, sarcasm, or harmful intent.",
            "certainty": "detected"
        }

    elif verdict == "offensive":
        consider("Backhanded compliment", BACKHANDED)
        consider("Negative sarcasm (person-targeted)", NEG_SARCASM)
        consider("Ironic praise", IRONIC_PRAISE)
        consider("Condescending dismissal", CONDESCENDING)
        consider("Rhetorical question as attack", RHETORICAL_ATK)
        consider("Hostile banter", HOSTILE_BANTER)

        if candidates:
            candidates.sort(key=lambda x: x[1], reverse=True)
            name, _, certainty = candidates[0]
            return {"type": name, "description": DESCRIPTIONS[name], "certainty": certainty}

        if h > 0.25:
            return {
                "type": "Borderline offensive / hateful",
                "description": "Sits at the edge between offensive and hate speech. Demeans a person or group but stops short of explicit targeting.",
                "certainty": "possible"
            }
        return {
            "type": "Direct insult",
            "description": "A straightforward demeaning statement without subtlety or ambiguity.",
            "certainty": "detected"
        }

    elif verdict == "hate":
        consider("Coded exclusionary language", CODED_EXCL)
        consider("Stereotype reinforcement", STEREOTYPE)
        consider("Dog-whistle language", DOG_WHISTLE, certainty="possible")

        if candidates:
            candidates.sort(key=lambda x: x[1], reverse=True)
            name, _, certainty = candidates[0]
            return {"type": name, "description": DESCRIPTIONS[name], "certainty": certainty}

        if h > 0.80:
            return {
                "type": "Targeted dehumanisation",
                "description": "Reduces a person or group to something less than human — through comparison, denial of traits, or explicit degradation.",
                "certainty": "detected"
            }
        return {
            "type": "Hate-leaning statement",
            "description": "Hostile framing toward a person or group, but below the threshold of explicit slurs or clear dehumanisation.",
            "certainty": "possible"
        }

    return {
        "type": "Unclassified",
        "description": "The model could not confidently assign a descriptive type to this sentence.",
        "certainty": "possible"
    }


# ── LLM-based type detection (primary path; falls back to detect_type_scored) ──
TYPE_CATEGORIES_BY_VERDICT = {
    "normal": ["Meta-commentary", "Situational sarcasm", "Self-deprecating humor",
               "Banter/playful ribbing", "Rhetorical/philosophical", "Neutral statement"],
    "offensive": ["Backhanded compliment", "Negative sarcasm (person-targeted)",
                  "Ironic praise", "Condescending dismissal",
                  "Rhetorical question as attack", "Hostile banter", "Direct insult"],
    "hate": ["Coded exclusionary language", "Stereotype reinforcement",
             "Dog-whistle language", "Targeted dehumanisation", "Hate-leaning statement"],
}

def detect_type_llm(text, verdict, scores):
    categories = TYPE_CATEGORIES_BY_VERDICT.get(verdict, [])
    prompt = f"""<start_of_turn>user
A sentence was classified as "{verdict}" by a toxicity model. Identify which descriptive category best explains its tone: {", ".join(categories)}

Sentence: "{text}"

Respond in this exact JSON format only, with all three fields present:
{{"type": "category name", "description": "one sentence explaining the tone, not just word presence", "certainty": "detected" or "possible"}}<end_of_turn>
<start_of_turn>model
"""
    try:
        inputs = gemma_tokenizer(prompt, return_tensors="pt").to(gemma_model.device)
        with torch.no_grad():
            output = gemma_model.generate(
                **inputs, max_new_tokens=100, do_sample=False,
                pad_token_id=gemma_tokenizer.eos_token_id,
            )
        generated = output[0][inputs["input_ids"].shape[1]:]
        raw = gemma_tokenizer.decode(generated, skip_special_tokens=True).strip()
        raw = re.sub(r"```json|```", "", raw).strip()
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if not match:
            return None
        result = json.loads(match.group())

        # Defensive validation — a malformed but parseable JSON object must not
        # crash the route downstream when desc["type"] etc. are accessed.
        type_val = result.get("type")
        desc_val = result.get("description")
        cert_val = result.get("certainty", "possible")
        if not type_val or not desc_val:
            return None
        if cert_val not in ("detected", "possible"):
            cert_val = "possible"
        return {"type": type_val, "description": desc_val, "certainty": cert_val}
    except Exception as e:
        print(f"Gemma type-detect error: {e}")
        return None


def should_call_type_llm(verdict, scores, confidence):
    """Gate the Gemma type-detect call so it only fires on cases the keyword
    fallback is genuinely weak at — ambiguous or low-confidence verdicts —
    rather than on every single request, which would tank latency."""
    h, o, n = scores["hate"], scores["offensive"], scores["normal"]
    if confidence < 0.75:
        return True
    if verdict == "normal" and (h + o) > 0.20:
        return True
    if verdict == "offensive" and h > 0.20:
        return True
    return False


from textnorm import normalize_text, has_surface_flags, looks_perturbed, SLANG_WHITELIST


# ── Load Gemma ────────────────────────────────────────────────
GEMMA_PATH = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/1"

print("Loading Gemma 2B-IT...")
gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH)
gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
gemma_model.eval()
print("Gemma loaded.")


def gemma_classify(text: str) -> dict | None:
    prompt = f"""<start_of_turn>user
Classify this sentence into exactly one of: hate speech, offensive, or normal.

Rules:
- Misspelled slurs (loooser, stuuupid) count the same as correctly spelled ones
- ALL CAPS aggressive phrasing toward a person = offensive
- Sarcasm at a situation = normal. Sarcasm at a person = offensive.
- If harsh but targeting no one = normal

Sentence: "{text}"

Respond in this exact JSON format only:
{{"verdict": "hate speech" or "offensive" or "normal", "confidence": 0.0-1.0, "reason": "one sentence"}}<end_of_turn>
<start_of_turn>model
"""
    try:
        inputs = gemma_tokenizer(prompt, return_tensors="pt").to(gemma_model.device)
        with torch.no_grad():
            output = gemma_model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=gemma_tokenizer.eos_token_id,
            )
        generated = output[0][inputs["input_ids"].shape[1]:]
        raw = gemma_tokenizer.decode(generated, skip_special_tokens=True).strip()
        raw = re.sub(r"```json|```", "", raw).strip()
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if not match:
            return None
        result = json.loads(match.group())
        verdict_map = {"hate speech": "hate", "offensive": "offensive", "normal": "normal"}
        verdict = verdict_map.get(result.get("verdict", "").lower(), "normal")
        conf = float(result.get("confidence", 0.80))
        remaining = round((1 - conf) / 2, 4)
        all_scores = {"hate": remaining, "offensive": remaining, "normal": remaining}
        all_scores[verdict] = conf
        return {
            "verdict":          verdict,
            "confidence":       float(result.get("confidence", 0.80)),
            "scores": all_scores,
            "tone":             result.get("reason", "LLM classification"),
            "desc_type":        "LLM-routed classification",
            "desc_text":        result.get("reason", ""),
            "desc_certainty":   "detected",
            "reason":           f"Routed to Gemma. {result.get('reason', '')}",
            "word_importances": []
        }
    except Exception as e:
        print(f"Gemma error: {e}")
        return None



def llm_classify(text: str) -> dict | None:
    return gemma_classify(text)


# -- Occlusion-based word importance (leave-one-out; Li, Monroe & Jurafsky 2016) --
def get_word_importances(text, pred_label_idx, top_n=8):
    words = text.split()
    if len(words) < 2:
        return []

    def get_prob(t):
        enc = tokenizer(t, return_tensors="pt", truncation=True,
                        padding=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(**enc).logits, dim=-1).cpu().numpy()[0]
        return probs[pred_label_idx]

    baseline = get_prob(text)
    importances = []
    for i, word in enumerate(words):
        if not word.strip(".,!?;:\"'"):
            continue
        masked = " ".join(w for j, w in enumerate(words) if j != i)
        if not masked.strip():
            continue
        importances.append({
            "word": word,
            "score": round(float(baseline - get_prob(masked)), 4)
        })

    importances.sort(key=lambda x: abs(x["score"]), reverse=True)
    return importances[:top_n]

def check_self_distress_llm(text):
    prompt = f"""<start_of_turn>user
Does this sentence express self-directed hopelessness, a wish to not exist, feeling like one doesn't matter, or passive suicidal ideation — directed at the speaker themselves, not at someone else?

If the sentence is aimed at another person (you / they / them) - an insult, a threat, exclusion, or criticism - answer false, even if it sounds bleak.

Sentence: "{text}"

Respond in this exact JSON format only:
{{"distress": true or false, "confidence": 0.0-1.0}}<end_of_turn>
<start_of_turn>model
"""
    try:
        inputs = gemma_tokenizer(prompt, return_tensors="pt").to(gemma_model.device)
        with torch.no_grad():
            output = gemma_model.generate(
                **inputs, max_new_tokens=40, do_sample=False,
                pad_token_id=gemma_tokenizer.eos_token_id,
            )
        generated = output[0][inputs["input_ids"].shape[1]:]
        raw = gemma_tokenizer.decode(generated, skip_special_tokens=True).strip()
        raw = re.sub(r"```json|```", "", raw).strip()
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if not match:
            return False
        result = json.loads(match.group())
        return bool(result.get("distress")) and float(result.get("confidence", 0)) > 0.5
    except Exception as e:
        print(f"Gemma distress-check error: {e}")
        return False

# ── Reason narrative ──────────────────────────────────────────
def build_reason(verdict, scores, word_importances, desc_type):
    h, o, n = scores["hate"], scores["offensive"], scores["normal"]
    pushing = [w["word"] for w in word_importances if w["score"] > 0.01][:3]
    pulling = [w["word"] for w in word_importances if w["score"] < -0.01][:2]

    parts = [f"Classified as <strong>{verdict}</strong>."]

    if pushing:
        parts.append(
            f'The word{"s" if len(pushing)>1 else ""} '
            f'<em>{", ".join(pushing)}</em> '
            f'carried the most weight toward this verdict.'
        )
    if pulling:
        parts.append(
            f'<em>{", ".join(pulling)}</em> pulled slightly against it.')

    top = max(h, o, n)
    if top > 0.90:
        parts.append("The model is highly confident.")
    elif top > 0.70:
        parts.append("Moderate confidence — most signals align.")
    else:
        parts.append("Low confidence — sits near a decision boundary. Different phrasing could shift the verdict.")

    second_name = sorted(LABEL_IDX.keys(), key=lambda k: scores[k], reverse=True)[1]
    second_val  = scores[second_name]
    if second_val > 0.25:
        parts.append(
            f'A {second_val*100:.0f}% signal toward <strong>{second_name}</strong> '
            f'suggests the sentence also carries some characteristics of that class.'
        )

    return " ".join(parts)


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/classify", methods=["POST"])
def classify():
    data = request.get_json()
    text = data.get("text", "").strip()
    original_text = text
    # normalize only when the input looks perturbed - unconditional normalization is
    # net-negative on a human-crafted functional suite (see NORMALIZATION RECOVERY /
    # HateCheck spelling table).
    text, corrections = normalize_text(text) if looks_perturbed(text) else (text, [])
    if not text:
        return jsonify({"error": "No text provided"}), 400

    # Stage 1: surface flags BERT structurally can't handle
    flagged, flag_reason = has_surface_flags(text)
    if flagged:
        llm_result = llm_classify(text)
        if llm_result:
            return _respond(llm_result, text)
        flag_reason = f"surface flag ({flag_reason}) — LLM unavailable, BERT used as fallback"

    # Stage 2: BERT inference
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(**inputs).logits / TEMPERATURE, dim=-1).cpu().numpy()[0]

    scores     = {LABELS[i]: round(float(probs[i]), 4) for i in range(3)}
    pred_idx   = int(probs.argmax())
    verdict    = LABELS[pred_idx]
    confidence = round(float(probs[pred_idx]), 4)

    if confidence < ROUTE_THRESHOLD:
        llm_result = llm_classify(text)
        if llm_result:
            verdict    = llm_result["verdict"]
            scores     = llm_result["scores"]
            confidence = llm_result["confidence"]
            word_importances = []  # Gemma path has no occlusion pass
            desc = None
            if should_call_type_llm(verdict, scores, confidence):
                desc = detect_type_llm(text, verdict, scores)
            if desc is None:
                desc = detect_type_scored(verdict, scores, text, word_importances)

            tone = get_tone(scores["hate"], scores["offensive"], scores["normal"])
            reason = f"Routed to Gemma ({llm_result.get('desc_text','')}). " + build_reason(verdict, scores, word_importances, desc)

            return _respond({
                "verdict": verdict,
                "confidence": confidence,
                "scores": scores,
                "tone": tone,
                "desc_type": desc["type"],
                "desc_text": desc["description"],
                "desc_certainty": desc["certainty"],
                "reason": reason,
                "word_importances": word_importances,
                "corrections": corrections,
                "original_text": original_text if corrections else None,
            }, text)

    word_importances = get_word_importances(text, pred_idx)

    # Stage 4: descriptive type — Gemma primary (gated), keyword scored fallback
    tone = get_tone(scores["hate"], scores["offensive"], scores["normal"])
    desc = None
    if should_call_type_llm(verdict, scores, confidence):
        desc = detect_type_llm(text, verdict, scores)
    if desc is None:
        desc = detect_type_scored(verdict, scores, text, word_importances)

    reason = build_reason(verdict, scores, word_importances, desc)

    if flagged:
        reason = f"⚠️ {flag_reason}. " + reason

    return _respond({
        "verdict":          verdict,
        "confidence":       confidence,
        "scores":           scores,
        "tone":             tone,
        "desc_type":        desc["type"],
        "desc_text":        desc["description"],
        "desc_certainty":   desc["certainty"],
        "reason":           reason,
        "word_importances": word_importances,
        "corrections":      corrections,
        "original_text":    original_text if corrections else None,
    }, text)


if __name__ == "__main__":
    app.run(debug=True, port=5000)

In [ ]:
import os
os.makedirs("templates", exist_ok=True)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Toxicity Classifier</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;500&display=swap');

  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

  :root {
    --bg:        #0f0f11;
    --surface:   #18181c;
    --surface2:  #1e1e24;
    --border:    #2a2a30;
    --text:      #e8e8ec;
    --muted:     #6b6b78;
    --normal:    #4ade80;
    --offensive: #fb923c;
    --hate:      #f87171;
    --accent:    #818cf8;
  }

  body {
    background: var(--bg);
    color: var(--text);
    font-family: 'IBM Plex Sans', sans-serif;
    font-weight: 300;
    min-height: 100vh;
    display: flex;
    flex-direction: column;
    align-items: center;
    padding: 48px 24px 100px;
  }

  header { width: 100%; max-width: 720px; margin-bottom: 40px; }

  .eyebrow {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.15em;
    color: var(--accent);
    text-transform: uppercase;
    margin-bottom: 12px;
  }

  h1 {
    font-size: clamp(26px, 5vw, 40px);
    font-weight: 300;
    letter-spacing: -0.02em;
    line-height: 1.15;
  }
  h1 span { color: var(--accent); }

  .subtitle {
    margin-top: 10px;
    font-size: 14px;
    color: var(--muted);
    line-height: 1.6;
  }

  .card {
    width: 100%;
    max-width: 720px;
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 24px 28px;
    margin-bottom: 14px;
  }

  textarea {
    width: 100%;
    background: var(--bg);
    border: 1px solid var(--border);
    border-radius: 8px;
    color: var(--text);
    font-family: 'IBM Plex Sans', sans-serif;
    font-size: 15px;
    font-weight: 300;
    line-height: 1.6;
    padding: 14px 16px;
    resize: vertical;
    min-height: 90px;
    outline: none;
    transition: border-color 0.15s;
  }
  textarea::placeholder { color: var(--muted); }
  textarea:focus { border-color: var(--accent); }

  .actions { display: flex; justify-content: flex-end; margin-top: 10px; }

  button {
    background: var(--accent);
    color: #0f0f11;
    border: none;
    border-radius: 7px;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 13px;
    font-weight: 600;
    letter-spacing: 0.04em;
    padding: 10px 24px;
    cursor: pointer;
    transition: opacity 0.15s, transform 0.1s;
  }
  button:hover  { opacity: 0.88; }
  button:active { transform: scale(0.97); }
  button:disabled { opacity: 0.4; cursor: not-allowed; }

  #results { width: 100%; max-width: 720px; display: none; }

  /* Two-column layout for verdict + type */
  .top-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 14px;
    margin-bottom: 14px;
  }
  @media (max-width: 560px) { .top-grid { grid-template-columns: 1fr; } }

  .section-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 14px;
  }

  /* Verdict */
  .verdict-row { display: flex; align-items: center; gap: 12px; margin-bottom: 6px; }

  .verdict-badge {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    padding: 4px 10px;
    border-radius: 4px;
    border: 1px solid;
  }
  .verdict-badge.normal    { color: var(--normal);    border-color: var(--normal);    background: #4ade8012; }
  .verdict-badge.offensive { color: var(--offensive); border-color: var(--offensive); background: #fb923c12; }
  .verdict-badge.hate      { color: var(--hate);      border-color: var(--hate);      background: #f8717112; }

  .confidence-text {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 13px;
    color: var(--muted);
  }

  .tone-pill {
    font-size: 12px;
    color: var(--muted);
    font-style: italic;
    margin-top: 8px;
  }

  /* Bars */
  .bars { display: flex; flex-direction: column; gap: 12px; }

  .bar-row {
    display: grid;
    grid-template-columns: 82px 1fr 52px;
    align-items: center;
    gap: 12px;
  }

  .bar-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: var(--muted);
    text-align: right;
  }

  .bar-track { height: 5px; background: var(--border); border-radius: 3px; overflow: hidden; }

  .bar-fill {
    height: 100%;
    border-radius: 3px;
    width: 0%;
    transition: width 0.55s cubic-bezier(0.25, 1, 0.5, 1);
  }
  .bar-fill.normal    { background: var(--normal); }
  .bar-fill.offensive { background: var(--offensive); }
  .bar-fill.hate      { background: var(--hate); }

  .bar-pct { font-family: 'IBM Plex Mono', monospace; font-size: 12px; color: var(--muted); text-align: right; }

  .divider { border: none; border-top: 1px solid var(--border); margin: 18px 0; }

  /* ── Sentence type card ── */
  .type-header {
    display: flex;
    align-items: center;
    gap: 10px;
    margin-bottom: 12px;
  }

  .type-name {
    font-size: 15px;
    font-weight: 500;
    color: var(--text);
  }

  .certainty-badge {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    padding: 2px 7px;
    border-radius: 3px;
    border: 1px solid;
  }
  .certainty-badge.detected { color: var(--normal);  border-color: #4ade8040; background: #4ade8010; }
  .certainty-badge.possible { color: var(--accent);  border-color: #818cf840; background: #818cf810; }

  .type-desc {
    font-size: 13px;
    color: #b0b0bc;
    line-height: 1.7;
  }

  /* ── Word chips ── */
  .words-wrap { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 4px; }

  .word-chip {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 12px;
    padding: 3px 9px;
    border-radius: 4px;
    border: 1px solid;
  }
  .word-chip.push { color: var(--normal);    border-color: #4ade8040; background: #4ade8010; }
  .word-chip.pull { color: var(--hate);      border-color: #f8717140; background: #f8717110; }

  .chip-score { font-size: 10px; opacity: 0.6; margin-left: 4px; }

  .chip-legend { display: flex; gap: 16px; margin-top: 12px; }
  .chip-legend span { font-size: 11px; color: var(--muted); display: flex; align-items: center; gap: 5px; }
  .dot { width: 8px; height: 8px; border-radius: 50%; display: inline-block; }
  .dot.push { background: var(--normal); }
  .dot.pull { background: var(--hate); }

  /* ── Reason ── */
  .reason-box {
    font-size: 13px;
    line-height: 1.75;
    color: #b0b0bc;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 14px 16px;
  }
  .reason-box strong { color: var(--text); font-weight: 500; }
  .reason-box em     { color: var(--accent); font-style: normal; }

  /* Spinner */
  .spinner {
    display: inline-block;
    width: 13px; height: 13px;
    border: 2px solid #0f0f1160;
    border-top-color: #0f0f11;
    border-radius: 50%;
    animation: spin 0.7s linear infinite;
    vertical-align: middle;
    margin-right: 6px;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  .error-msg {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 12px;
    color: var(--hate);
    margin-top: 8px;
    display: none;
  }
</style>
</head>
<body>

<header>
  <p class="eyebrow">BERT · HateXplain · Fine-tuned</p>
  <h1>Does this cross<br>the <span>line?</span></h1>
  <p class="subtitle">Paste any sentence and see where it sits — plus what kind of sentence the model thinks it is and why.</p>
</header>

<div class="card">
  <textarea id="inputText" placeholder="Type or paste a sentence…" rows="3"></textarea>
  <div class="actions">
    <button id="classifyBtn" onclick="classify()">Analyse →</button>
  </div>
  <p class="error-msg" id="errorMsg">Something went wrong. Try again.</p>
</div>

<div id="results">

  <div id="analysisBlocks">
  <!-- Top row: verdict + sentence type side by side -->
  <div class="top-grid">

    <!-- Verdict card -->
    <div class="card" style="margin-bottom:0">
      <p class="section-label">Verdict</p>
      <div class="verdict-row">
        <span class="verdict-badge" id="verdictBadge">—</span>
        <span class="confidence-text" id="confidenceText"></span>
      </div>
      <p class="tone-pill" id="toneText"></p>

      <hr class="divider">

      <p class="section-label">Probability breakdown</p>
      <div class="bars">
        <div class="bar-row">
          <span class="bar-label">Normal</span>
          <div class="bar-track"><div class="bar-fill normal" id="barNormal"></div></div>
          <span class="bar-pct" id="pctNormal">—</span>
        </div>
        <div class="bar-row">
          <span class="bar-label">Offensive</span>
          <div class="bar-track"><div class="bar-fill offensive" id="barOffensive"></div></div>
          <span class="bar-pct" id="pctOffensive">—</span>
        </div>
        <div class="bar-row">
          <span class="bar-label">Hate</span>
          <div class="bar-track"><div class="bar-fill hate" id="barHate"></div></div>
          <span class="bar-pct" id="pctHate">—</span>
        </div>
      </div>
    </div>

    <!-- Sentence type card -->
    <div class="card" style="margin-bottom:0">
      <p class="section-label">Sentence type</p>
      <div class="type-header">
        <span class="type-name" id="typeName">—</span>
        <span class="certainty-badge" id="certaintyBadge">—</span>
      </div>
      <p class="type-desc" id="typeDesc"></p>
    </div>

  </div>

  <!-- Word influence -->
  <div class="card">
    <p class="section-label">Word influence</p>
    <div class="words-wrap" id="wordChips"></div>
    <div class="chip-legend">
      <span><span class="dot push"></span> pushed toward verdict</span>
      <span><span class="dot pull"></span> pulled against it</span>
    </div>
  </div>

  <!-- Why this verdict -->
  <div class="card">
    <p class="section-label">Why this verdict</p>
    <div class="reason-box" id="reasonBox"></div>
  </div>
  </div><!-- analysisBlocks -->

  <div id="supportNotice" class="card" style="display:none">
    <p class="section-label">Support</p>
    <h3 id="noticeTitle" style="margin:4px 0 10px;font-weight:500;font-size:1rem"></h3>
    <p id="noticeBody" class="type-desc"></p>
    <ul id="noticeResources" style="margin:14px 0 0;padding-left:18px;line-height:1.75"></ul>
    <p id="noticeDisclaimer" class="type-desc" style="margin-top:14px;opacity:.65"></p>
  </div>


</div>

<script>
async function classify() {
  const text = document.getElementById("inputText").value.trim();
  if (!text) return;

  const btn = document.getElementById("classifyBtn");
  const err = document.getElementById("errorMsg");
  err.style.display = "none";
  document.getElementById("results").style.display = "none";

  btn.disabled = true;
  btn.innerHTML = '<span class="spinner"></span>Analysing';

  try {
    const res = await fetch("/classify", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ text })
    });
    if (!res.ok) throw new Error();
    const d = await res.json();

    // Support resource: shown ALONGSIDE the normal analysis, never replacing it.
    const noticeEl = document.getElementById("supportNotice");
    const sr = d.support_resource;
    if (sr) {
      document.getElementById("noticeTitle").textContent = sr.title || "";
      document.getElementById("noticeBody").textContent  = sr.body  || "";
      const ul = document.getElementById("noticeResources");
      ul.innerHTML = "";
      (sr.resources || []).forEach(r => {
        const li = document.createElement("li");
        const link = r.url
          ? ` \u00b7 <a href="${r.url}" target="_blank" rel="noopener">${r.url.replace(/^https?:\/\//, "")}</a>`
          : "";
        li.innerHTML = `<strong>${r.name}</strong> \u2014 ${r.contact}` + link;
        ul.appendChild(li);
      });
      document.getElementById("noticeDisclaimer").textContent = sr.disclaimer || "";
      noticeEl.style.display = "block";
    } else {
      noticeEl.style.display = "none";
    }


    // Verdict
    const badge = document.getElementById("verdictBadge");
    badge.textContent = d.verdict.toUpperCase();
    badge.className   = "verdict-badge " + d.verdict;
    document.getElementById("confidenceText").textContent =
      (d.confidence * 100).toFixed(1) + "% confidence";
    document.getElementById("toneText").textContent = d.tone;

    // Bars
    ["barNormal","barOffensive","barHate"].forEach(id =>
      document.getElementById(id).style.width = "0%"
    );
    requestAnimationFrame(() => requestAnimationFrame(() => {
      setBar("barNormal",    "pctNormal",    d.scores.normal);
      setBar("barOffensive", "pctOffensive", d.scores.offensive);
      setBar("barHate",      "pctHate",      d.scores.hate);
    }));

    // Sentence type
    document.getElementById("typeName").textContent = d.desc_type;
    document.getElementById("typeDesc").textContent = d.desc_text;
    const cb = document.getElementById("certaintyBadge");
    cb.textContent = d.desc_certainty === "detected" ? "✓ detected" : "~ possible";
    cb.className   = "certainty-badge " + d.desc_certainty;

    // Word chips
    const container = document.getElementById("wordChips");
    container.innerHTML = "";
    (d.word_importances || []).forEach(w => {
      const chip = document.createElement("span");
      chip.className = "word-chip " + (w.score >= 0 ? "push" : "pull");
      const sign = w.score >= 0 ? "+" : "";
      chip.innerHTML = `${w.word}<span class="chip-score">${sign}${(w.score*100).toFixed(1)}%</span>`;
      container.appendChild(chip);
    });

    // Reason
    document.getElementById("reasonBox").innerHTML = d.reason;

    document.getElementById("results").style.display = "block";

  } catch(e) {
    err.style.display = "block";
  } finally {
    btn.disabled = false;
    btn.innerHTML = "Analyse →";
  }
}

function setBar(barId, pctId, score) {
  document.getElementById(barId).style.width = (score * 100).toFixed(1) + "%";
  document.getElementById(pctId).textContent = (score * 100).toFixed(1) + "%";
}

document.getElementById("inputText").addEventListener("keydown", e => {
  if (e.key === "Enter" && (e.ctrlKey || e.metaKey)) classify();
});
</script>
</body>
</html>

In [ ]:
!pip install pyngrok

In [ ]:
import subprocess, threading
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# Kill anything on port 5000
subprocess.run(["fuser", "-k", "5000/tcp"], capture_output=True)

# ngrok token from Kaggle Secrets (Add-ons -> Secrets, name it "ngrok_token").
# The previously hard-coded token was committed to a shared file -> revoke it in the
# ngrok dashboard and paste a fresh one into the secret.
ngrok.set_auth_token(UserSecretsClient().get_secret("ngrok_token"))

# Start Flask in background
t = threading.Thread(target=lambda: subprocess.run(["python", "app.py"]))
t.daemon = True
t.start()

import time
time.sleep(3)  # give Flask time to start

# Expose publicly
public_url = ngrok.connect(5000)
print("Your app is live at:", public_url)